# ECGHybridV15 — SEM CORAL + SEM LOESS+NLM (Ablação #1) — **Patient-Level Split**

**Ablação:** LOESS e NLM removidos — apenas Butterworth (0.5–40 Hz).

**Revisão em resposta ao Revisor 4 (IEEE Access):**

Este notebook é uma versão adaptada do `ablation_loess_nlm.ipynb` original, com
**a única alteração de metodologia** sendo a substituição do split aleatório-estratificado
por um split **patient-aware** que garante exclusividade por paciente entre treino,
calibração e holdout, bem como entre folds do 10-fold CV interno.

**O que muda:**
- `train_test_split` → `StratifiedGroupKFold(n_splits=7)` com `groups=patient_id` para o holdout (~14,3%)
- `train_test_split` → `StratifiedGroupKFold(n_splits=17)` com `groups=patient_id[trainval]` para a calibração INT8 (1/17 do trainval ≈ 5,0% do pool)
- `StratifiedKFold(n_splits=10)` → `StratifiedGroupKFold(n_splits=10)` com `groups=patient_id[train]` no CV interno

**Correção adicional (Cell 35):**
- Winsorização do representative dataset INT8 (`clip=p99.9` do treino) para
  evitar que outliers de artefato (ex.: eletrodo desconectado) inflem os
  scales de quantização e destruam a resolução INT8 do ramo RAW.

**O que NÃO muda:**
- SEED=42, arquitetura, hiperparâmetros, Focal Loss, augmentation, threshold Youden,
  pipeline PyTorch→Keras→LiteRT INT8, threshold recalibrado, deployment no XIAO ESP32-S3.

**Grupos:**
- **PTB-XL:** `patient_id` real do `ptbxl_database.csv`
- **CSN:** um grupo sintético único por registro (Zheng et al. 2020)

**Saída:** `/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM/`

**Versão v2 (padronização final):** rótulos corrigidos (split patient-aware), `ECG_MODEL_VARIANT` no header C e célula final padronizada que gera, a partir das variáveis, as métricas INT8 com IC 95% bootstrap, o `metrics_final.json`, o `holdout_probs_int8.csv` (referência sinal a sinal para o ESP32-S3) e a ROC INT8.

**Versão v3 (treino determinístico):** `cudnn.deterministic=True`, `cudnn.benchmark=False`, `torch.use_deterministic_algorithms(True)`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, semente reiniciada no início de cada fold (`SEED + fold`) e `DataLoader` com `generator` e `worker_init_fn` fixos. Header C com τ em 8 casas e limiar inteiro `ECG_THRESHOLD_Q`. **Executar sempre com kernel reiniciado (Restart & Run All).**

## 1. Imports e configuracao

In [1]:
# ── Determinismo (v3): deve ser definido ANTES do primeiro import do torch ──
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

VARIANT_LABEL = 'Ablation #1 - No-LOESS/NLM (Butterworth only)'
VARIANT_ID    = 'ABL1_NO_LOESS_NLM'

import os, pickle, glob, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, roc_curve, auc as sk_auc)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# ── Determinismo (v3) ───────────────────────────────────────────────────
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.use_deterministic_algorithms(True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)

set_seed(SEED)
print(f'torch {torch.__version__} | cudnn.deterministic=True | '
      f'use_deterministic_algorithms=True | CUBLAS_WORKSPACE_CONFIG={os.environ["CUBLAS_WORKSPACE_CONFIG"]}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

FS = 500; N_POINTS = 5000; N_ACF_LAGS = 5000
N_FOLDS      = 10
BATCH_SIZE   = 64
EPOCHS       = 80
LR_INITIAL   = 4e-3
PATIENCE     = 15
HOLDOUT_FRAC = 0.15   # alvo nominal; fração real ≈14,3% (StratifiedGroupKFold, 7 folds)
CALIB_FRAC   = 0.05   # 5%  calibração INT8 (split formal, separado antes do K-Fold)

# ── Splits patient-aware ──────────────────────────────────────────────
# Number of folds chosen to approximate the original 15%/5% fractions:
#   7 folds  → 1 fold ≈ 14.3% (holdout)
#   17 folds → 1 fold ≈ 5.9% do trainval ≈ 5.0% do pool (calibração INT8)
N_SPLITS_HOLDOUT = 7
N_SPLITS_CALIB   = 17

AUG_PROB = 0.30; AUG_SNR_MIN = 20; AUG_SNR_MAX = 40; AUG_AMP_RANGE = 0.15
FOCAL_GAMMA = 2.0; FOCAL_ALPHA = 0.80

# ── NOVO diretório de saída (não sobrescreve o original) ──────────────
BASE = '/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3/'
os.makedirs(BASE, exist_ok=True)
FINAL_MODEL_PATH  = f'{BASE}/ablation_loess_nlm_final.pt'
SCALER_PATH       = f'{BASE}/ablation_loess_nlm_scaler.pkl'
THRESH_PATH       = f'{BASE}/ablation_loess_nlm_threshold.pkl'
TFLITE_PATH       = f'{BASE}/ablation_loess_nlm_int8.tflite'
C_HEADER_PATH     = f'{BASE}/ablation_loess_nlm_model.h'
HOLDOUT_CSV_PATH  = f'{BASE}/ablation_loess_nlm_holdout.csv'
CALIB_IDX_PATH    = f'{BASE}/ablation_loess_nlm_calib_indices.pkl'
ROC_PATH          = f'{BASE}/ablation_loess_nlm_roc.png'
RESULTS_PATH      = f'{BASE}/ablation_loess_nlm_results.png'
WAVE_PLOT_PATH    = f'{BASE}/ablation_loess_nlm_waves.png'

# ── Paths dos datasets ─────────────────────────────────────────────────
# Os sinais serao filtrados apenas com Butterworth (HP 0.5 Hz / LP 40 Hz)
# Ablation: LOESS e NLM removidos
PATHS_CH = {
    'AFIB': '/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_AFIB/',
    'SR':   '/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_SR/',
}
PATHS_PTB = {
    'AFIB': '/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_AFIB/',
    'SR':   '/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_SR/',
}

# ── Metadados PTB-XL (mapeamento ecg_id → patient_id) ─────────────────
PTBXL_DB_CSV = '/mnt/windows/Doutorado/Datasets/PTB_XL_Raw/ptbxl_database.csv'

C_AFIB = '#D85A30'; C_SR = '#185FA5'
print(f'HOLDOUT_FRAC : {HOLDOUT_FRAC}  (alvo nominal; real ≈14,3% patient-aware)')
print(f'CALIB_FRAC   : {CALIB_FRAC}  ({int(CALIB_FRAC*100)}% calibracao INT8 formal)')
print(f'SPLITS       : StratifiedGroupKFold — holdout={N_SPLITS_HOLDOUT} folds, calib={N_SPLITS_CALIB} folds')
print(f'  CH  AFIB -> {PATHS_CH["AFIB"]}')
print(f'  CH  SR   -> {PATHS_CH["SR"]}')
print(f'  PTB AFIB -> {PATHS_PTB["AFIB"]}')
print(f'  PTB SR   -> {PATHS_PTB["SR"]}')
print(f'  PTB-XL DB -> {PTBXL_DB_CSV}')
print(f'  BASE      -> {BASE}')

torch 2.12.0.dev20260304+cu128 | cudnn.deterministic=True | use_deterministic_algorithms=True | CUBLAS_WORKSPACE_CONFIG=:4096:8
Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Ti
HOLDOUT_FRAC : 0.15  (alvo nominal; real ≈14,3% patient-aware)
CALIB_FRAC   : 0.05  (5% calibracao INT8 formal)
SPLITS       : StratifiedGroupKFold — holdout=7 folds, calib=17 folds
  CH  AFIB -> /mnt/windows/Doutorado/Datasets/Chinese_Lead_I_AFIB/
  CH  SR   -> /mnt/windows/Doutorado/Datasets/Chinese_Lead_I_SR/
  PTB AFIB -> /mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_AFIB/
  PTB SR   -> /mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_SR/
  PTB-XL DB -> /mnt/windows/Doutorado/Datasets/PTB_XL_Raw/ptbxl_database.csv
  BASE      -> /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3/


## 2. Funcoes auxiliares

## 2b. Pipeline de Filtros Python - replica do firmware ESP32-S3 (ABLATION SEM LOESS+NLM)

**LOESS Smoothing e NLM Denoising REMOVIDOS** nesta ablação. Apenas Butterworth bandpass.

| Estagio | Arquivo C              | Status                                              |
|---------|------------------------|-----------------------------------------------------|
| 1       | `butterworth_filter.c` | Passa-banda zero-phase: HP 0.5 Hz + LP 40 Hz, fs=500 Hz |
| 2       | `loess_filter.c`       | **REMOVIDO** nesta ablação                          |
| 3       | `nlm_filter.c`         | **REMOVIDO** nesta ablação                          |


In [2]:
# =============================================================================
# PIPELINE DE FILTROS - replica Python do firmware ESP32-S3
# Referencias: butterworth_filter.c, loess_filter.c, nlm_filter.c
# =============================================================================

from scipy.signal import sosfilt
import math as _math

# ── 1. Butterworth bandpass zero-phase ────────────────────────────────────────
#
# Replica butterworth_filter.c:
#   - 2a ordem Butterworth HP 0.5 Hz cascateado com LP 40 Hz
#   - Forward-backward (zero-phase) - equivalente a filtfilt de 4a ordem
#
# Coeficientes calculados pela mesma BLT com pre-warping de frequencia
# (compute_hp_coeffs / compute_lp_coeffs). sosfilt two-pass manual
# com reset de estado entre passes replica dsps_biquad_f32 forward+backward.

def _butter_sos_blt(fc, btype, fs=500.0):
    """Coeficientes SOS via BLT com pre-warping - replica compute_hp/lp_coeffs."""
    omega_a = 2.0 * fs * _math.tan(_math.pi * fc / fs)
    k  = (2.0 * fs) / omega_a
    k2 = k * k
    sqrt2k = _math.sqrt(2.0) * k
    a0 = k2 + sqrt2k + 1.0
    if btype == 'high':
        b0 =  k2 / a0
        b1 = -2.0 * k2 / a0
        b2 =  k2 / a0
    else:  # 'low'
        b0 = 1.0 / a0
        b1 = 2.0 / a0
        b2 = 1.0 / a0
    a1 =  2.0 * (1.0 - k2) / a0
    a2 = (k2 - sqrt2k + 1.0) / a0
    # Formato SOS scipy: [b0, b1, b2, 1, a1, a2]
    return np.array([[b0, b1, b2, 1.0, a1, a2]], dtype=np.float64)

_SOS_HP = _butter_sos_blt(0.5,  'high', fs=500.0)  # pre-computado
_SOS_LP = _butter_sos_blt(40.0, 'low',  fs=500.0)  # pre-computado

def butterworth_apply(signal):
    """Bandpass zero-phase: HP 0.5 Hz -> LP 40 Hz, forward-backward.

    Replica butterworth_filter_apply() com 4 chamadas a dsps_biquad_f32:
      Forward : HP(signal) -> scratch -> LP(scratch) -> signal
      Reverse : signal[::-1]
      Backward: HP -> LP (mesmo esquema, state zerado)
      Reverse : restaura ordem causal
    """
    s = signal.astype(np.float64).copy()

    def _one_pass(x):
        # state zerado a cada pass (replica memset do firmware)
        y1 = sosfilt(_SOS_HP, x)
        y2 = sosfilt(_SOS_LP, y1)
        return y2

    s = _one_pass(s)           # Forward
    s = s[::-1].copy()         # Reverse
    s = _one_pass(s)           # Backward
    s = s[::-1].copy()         # Restore order
    return s.astype(np.float32)


# ── 2. LOESS robusto ──────────────────────────────────────────────────────────
#
# Replica loess_filter.c:
#   LOESS_SPAN        = 151 amostras (half_span = 75)
#   Ajuste local de reta (1a ordem) por WLS com peso tricube
#   2 passes: 1o sem mascara, 2o com mascara MAD de outliers
#   Subtrai a tendencia estimada do sinal (baseline correction)
#   LOESS_OUTLIER_SIGMA = 6.0  (sem fator de consistencia no threshold)

LOESS_SPAN      = 151
LOESS_HALF_SPAN = (LOESS_SPAN - 1) // 2   # 75
LOESS_OUTLIER_S = 6.0

def _tricube(u):
    """w(u) = (1 - |u|^3)^3  para |u| < 1 - replica tricube() em C."""
    u = np.abs(u)
    out = np.zeros_like(u)
    m = u < 1.0
    t = 1.0 - u[m] ** 3
    out[m] = t ** 3
    return out

def _loess_fit_at(signal, i, outlier_mask=None):
    """Ajuste linear local WLS no ponto i - replica loess_fit_at().

    Parametros do firmware:
      start/end clampados nas bordas
      max_dist = max(i - start, end - i, 1)
      u = |j - i| / max_dist
      w = tricube(u)
    Retorna beta_0 (intercepto = valor ajustado em t=0).
    """
    N = len(signal)
    start = max(0, i - LOESS_HALF_SPAN)
    end   = min(N - 1, i + LOESS_HALF_SPAN)
    max_dist = max(i - start, end - i, 1)

    j = np.arange(start, end + 1)
    if outlier_mask is not None:
        j = j[outlier_mask[j] == 0]
    if len(j) == 0:
        return signal[i]

    dist = np.abs(j - i) / max_dist
    w = _tricube(dist)
    t = (j - i).astype(np.float64)
    y = signal[j].astype(np.float64)

    sw   = np.sum(w)
    swx  = np.sum(w * t)
    swy  = np.sum(w * y)
    swxx = np.sum(w * t * t)
    swxy = np.sum(w * t * y)

    denom = sw * swxx - swx * swx
    if denom < 1e-9:
        return swy / sw if sw > 0 else signal[i]
    return (swy * swxx - swx * swxy) / denom

def loess_apply(signal):
    """Dois passes LOESS robusto - replica loess_filter_apply().

    Passe 1: ajuste sem mascara -> residuos -> MAD -> mascara
    Passe 2: reajuste com mascara
    Saida: signal - trend2  (baseline correction)
    """
    N = len(signal)
    sig = signal.astype(np.float64)

    # ── Passe 1 ──────────────────────────────────────────────────────────
    trend = np.array([_loess_fit_at(sig, i) for i in range(N)])
    residuals = sig - trend

    # Threshold MAD - replica exata do firmware (sem LOESS_MAD_FACTOR)
    # loess_filter.c usa: threshold = LOESS_OUTLIER_SIGMA * median(|residuals|)
    mad = np.median(np.abs(residuals))
    threshold = LOESS_OUTLIER_S * mad
    outlier_mask = (np.abs(residuals) > threshold).astype(np.uint8)

    # ── Passe 2 ──────────────────────────────────────────────────────────
    trend2 = np.array([_loess_fit_at(sig, i, outlier_mask) for i in range(N)])

    return (sig - trend2).astype(np.float32)


# ── 3. NLM (Non-Local Means) - Darbon fast ───────────────────────────────────
#
# Replica nlm_filter.c:
#   NLM_PATCH_HALF  = 13  (p)
#   NLM_SEARCH_HALF = 694 (M)
#   NLM_K_LAMBDA    = 1.5
#   NLM_MAD_FACTOR  = 1.4826
#   sigma_hat = MAD_FACTOR * median(|diff(x)|) / sqrt(2)
#   h = 2 * NLM_PATCH_LEN * lambda^2
#   Loop: D_delta[i] = (x[i+d]-x[i])^2, prefix sum, patch distance, exp(-d2/h)
#   Auto-similaridade (delta=0): w=1 por construcao

NLM_PATCH_HALF  = 13
NLM_PATCH_LEN   = 2 * NLM_PATCH_HALF + 1   # 27
NLM_SEARCH_HALF = 694
NLM_K_LAMBDA    = 1.5
NLM_MAD_FACTOR  = 1.4826

def _nlm_sigma(signal):
    """sigma_hat via MAD das primeiras diferencas - replica estimate_noise_sigma()."""
    diffs = np.abs(np.diff(signal.astype(np.float64)))
    return NLM_MAD_FACTOR * np.median(diffs) / np.sqrt(2.0)

def nlm_apply(signal):
    """NLM Darbon fast - replica nlm_filter_apply() — loop interno vetorizado.

    Logica identica ao firmware C e a versao escalar anterior:
      D_delta[i] = (signal[i+delta] - signal[i])^2
      cum_sum    = cumsum com padding de borda
      d2(i,j)    = cum[i+p] - cum[i-p-1]   (integral sobre janela de patch)
      w          = exp(-d2 / h)
      Acumulacao simetrica sobre todos os pares (i, i+delta) simultaneamente
    O loop `for i in range(N)` foi substituido por operacoes NumPy vetorizadas,
    mantendo resultado numericamente identico e reduzindo o tempo ~100-300x.
    """
    signal = signal.astype(np.float64)
    N = len(signal)
    p = NLM_PATCH_HALF
    M = NLM_SEARCH_HALF

    sigma_hat = _nlm_sigma(signal)
    lam = NLM_K_LAMBDA * sigma_hat
    h   = max(2.0 * NLM_PATCH_LEN * lam * lam, 1e-10)

    output = signal.copy()    # delta=0: self-similarity -> w=1
    norm   = np.ones(N)

    for delta in range(1, M + 1):
        valid_n = N - delta
        if valid_n <= 0:
            break

        # ── D_delta e prefix sum (identico ao firmware) ──────────────────
        diff_sq = (signal[delta:delta + valid_n] - signal[:valid_n]) ** 2
        cum = np.empty(N)
        cum[:valid_n] = np.cumsum(diff_sq)
        cum[valid_n:] = cum[valid_n - 1]

        # ── Indices vetorizados de todos os pares validos ─────────────────
        # i em [0, valid_n-1],  j = i + delta
        i_arr = np.arange(valid_n)          # shape (valid_n,)
        j_arr = i_arr + delta               # j < N por construcao

        # hi = min(i + p, valid_n - 1)
        hi_arr = np.minimum(i_arr + p, valid_n - 1)

        # lo = i - p - 1  (negativo -> 0 pelo operador ternario do firmware)
        lo_arr = i_arr - p - 1              # pode ser negativo

        # d2 = cum[hi] - (lo >= 0 ? cum[lo] : 0)
        cum_lo          = np.where(lo_arr >= 0, cum[lo_arr], 0.0)
        d2_arr          = cum[hi_arr] - cum_lo

        # w = exp(-d2 / h)  — vetorizado
        w_arr = np.exp(-d2_arr / h)

        # ── Acumulacao simetrica ──────────────────────────────────────────
        # output[i] += w * signal[j]
        np.add.at(output, i_arr, w_arr * signal[j_arr])
        np.add.at(norm,   i_arr, w_arr)
        # output[j] += w * signal[i]
        np.add.at(output, j_arr, w_arr * signal[i_arr])
        np.add.at(norm,   j_arr, w_arr)

    mask = norm > 1e-12
    return np.where(mask, output / norm, signal).astype(np.float32)


# ── 4. Pipeline completo ──────────────────────────────────────────────────────

def ecg_pipeline(signal):
    """Apenas Butterworth (LOESS e NLM REMOVIDOS) - ablation sem LOESS+NLM.

    Entrada: array 1-D float32, comprimento = N_POINTS
    Saida  : array 1-D float32, mesmo comprimento, filtrado
    """
    s = butterworth_apply(signal)
    # LOESS removido nesta ablation
    # NLM removido nesta ablation
    return s


# =============================================================================
# FUNCOES AUXILIARES (identicas a versao original)
# =============================================================================

def extract_acf(signal):
    """Autocorrelacao discreta normalizada - replica autocorr_compute().

    s = signal - mean(signal)
    R[k] = dot(s[:N-k], s[k:]) / dot(s, s)   k = 0..N-1
    """
    s = signal.flatten().astype(np.float64)
    n = len(s)
    s = s - s.mean()
    r0 = np.dot(s, s)
    if r0 < 1e-10:
        return np.zeros(n, dtype=np.float32)
    acf = np.empty(n, dtype=np.float32)
    for k in range(n):
        acf[k] = np.dot(s[:n - k], s[k:]) / r0
    return acf

def extract_acf_batch(X, desc=''):
    feats = []
    for i, sig in enumerate(X):
        if i % 200 == 0:
            print(f'  {desc} [{i}/{len(X)}]...', end='\r')
        feats.append(extract_acf(sig))
    print(f'  {desc} [{len(X)}/{len(X)}] OK          ')
    return np.array(feats, dtype=np.float32)

def load_domain(path_afib, path_sr, apply_filter=True):
    """Carrega CSVs de AFIB e SR, aplica pipeline de filtros se apply_filter=True.

    O pipeline (Butterworth -> LOESS -> NLM) e aplicado sinal a sinal antes
    de qualquer extracao de feature, replicando a sequencia do firmware.
    """
    X, y, fnames = [], [], []
    for path, label in [(path_afib, 1), (path_sr, 0)]:
        files = sorted(glob.glob(os.path.join(path, '*.csv')))
        if not files:
            raise RuntimeError(f'Nenhum CSV encontrado em: {path}')
        for fi, f in enumerate(files):
            if fi % 100 == 0:
                print(f'  Loading {os.path.basename(path)} [{fi}/{len(files)}]...', end='\r')
            try:
                s = pd.read_csv(f, header=None).values.flatten().astype(np.float32)
                s = s[:N_POINTS] if len(s) >= N_POINTS else np.pad(s, (0, N_POINTS - len(s)))
                if apply_filter:
                    s = ecg_pipeline(s)
                X.append(s)
                y.append(label)
                fnames.append(f)
            except Exception:
                continue
        print(f'  Loading {os.path.basename(path)} [{len(files)}/{len(files)}] OK   ')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), np.array(fnames)

print('Pipeline de filtros Python definido (ABLATION SEM LOESS+NLM).')
print(f'  Butterworth : HP=0.5 Hz, LP=40.0 Hz, fs=500, zero-phase (BLT)')
print(f'  LOESS       : ** REMOVIDO **')
print(f'  NLM Darbon  : ** REMOVIDO **')
print('Funcoes auxiliares (extract_acf, load_domain) definidas.')

# =============================================================================
# PATIENT-AWARE SPLIT — funções para construir vetor de patient_id por registro
# =============================================================================

_PTB_FN_RE = re.compile(r'^(\d+)_hr\.csv$')

def build_patient_groups(fnames, domain_flag, ptbxl_db_path=PTBXL_DB_CSV):
    """Constrói o vetor `groups` de identificadores de paciente para o pool combinado.
    
    - `domain_flag`: array com 0 (CH) ou 1 (PTB) por registro.
    - CSN (0): grupo sintético único por registro → 'CSN_<basename>'.
      (Zheng et al. 2020: cada arquivo do CSN corresponde a um paciente único.)
    - PTB-XL (1): grupo real 'PTB_<patient_id>' obtido de `ptbxl_database.csv`.
      O prefixo garante que IDs CSN e PTB nunca colidam.
    
    Retorna: np.array de strings (grupos), mesmo comprimento de `fnames`.
    """
    ptbxl = pd.read_csv(ptbxl_db_path, usecols=['ecg_id', 'patient_id'])
    ptbxl['ecg_id']     = ptbxl['ecg_id'].astype(int)
    ptbxl['patient_id'] = ptbxl['patient_id'].astype('Int64')
    ecg2pat = dict(zip(ptbxl['ecg_id'], ptbxl['patient_id']))

    groups = np.empty(len(fnames), dtype=object)
    unresolved = 0
    for i, (fpath, d) in enumerate(zip(fnames, domain_flag)):
        base = os.path.basename(fpath)
        if d == 0:
            groups[i] = f'CSN_{base}'
        else:
            m = _PTB_FN_RE.match(base)
            if m is None:
                groups[i] = f'PTB_ORPHAN_{base}'; unresolved += 1; continue
            pid = ecg2pat.get(int(m.group(1)))
            if pid is None or pd.isna(pid):
                groups[i] = f'PTB_ORPHAN_{base}'; unresolved += 1; continue
            groups[i] = f'PTB_{int(pid)}'
    if unresolved > 0:
        print(f'  ⚠️  {unresolved} arquivo(s) PTB sem patient_id resolvido — tratados como grupos únicos.')
    n_unique = len(set(groups))
    print(f'  Grupos únicos: {n_unique}  (para {len(groups)} registros)')
    return groups

def select_holdout_fold(splitter, X, y, groups):
    """Aplica um `Stratified*KFold` e retorna a fold cuja proporção AFIB mais
    se aproxima da proporção global — garante estratificação mesmo com grupos.
    Retorna (trainval_idx, holdout_idx).
    """
    frac_afib_global = float(y.sum()) / len(y)
    best = None; best_delta = 1.0
    for k, (tr_idx, ho_idx) in enumerate(splitter.split(X, y, groups=groups)):
        frac_ho = y[ho_idx].sum() / len(ho_idx)
        delta = abs(frac_ho - frac_afib_global)
        if delta < best_delta:
            best_delta = delta; best = (k, tr_idx, ho_idx, frac_ho)
    k, tr_idx, ho_idx, frac_ho = best
    print(f'  Fold escolhida: {k}  (fração AFIB holdout={frac_ho:.4f}, global={frac_afib_global:.4f})')
    return tr_idx, ho_idx

print('Funções patient-aware definidas: build_patient_groups(), select_holdout_fold().')


Pipeline de filtros Python definido (ABLATION SEM LOESS+NLM).
  Butterworth : HP=0.5 Hz, LP=40.0 Hz, fs=500, zero-phase (BLT)
  LOESS       : ** REMOVIDO **
  NLM Darbon  : ** REMOVIDO **
Funcoes auxiliares (extract_acf, load_domain) definidas.
Funções patient-aware definidas: build_patient_groups(), select_holdout_fold().


## 3. Carregamento e split patient-aware em três vias

O pool combinado CH + PTB-XL é dividido **por paciente** (`StratifiedGroupKFold`,
`groups = patient_id`), estratificado pela classe:

- **Holdout** (1 fold de 7, ≈14,3% do pool) → avaliado uma única vez, pós-treino
- **Calibração INT8** (1 fold de 17 do restante, ≈5,0% do pool) → representative dataset
- **Treino** (≈80,7% do pool) → 10-fold CV interno, também agrupado por paciente

Nenhum paciente aparece em mais de uma partição (verificado por `assert`).
Uma planilha `.csv` com todos os arquivos do holdout é salva ao final.

In [3]:
# =============================================================================
# SPLIT PATIENT-AWARE (Revisor 4) — Ablation LOESS/NLM
# =============================================================================
# Sinais carregados com ecg_pipeline() aplicando apenas Butterworth.
# Ablation: LOESS e NLM removidos — apenas Butterworth (0.5–40 Hz).
#
# Substitui `train_test_split` por `StratifiedGroupKFold` para garantir que
# nenhum paciente apareça em mais de uma partição (treino, calibração, holdout).
#
# Reprodutibilidade: SEED=42 preservado; splits determinísticos.

# ── Carrega os dois domínios ──────────────────────────────────────────────
print('Carregando CH...')
X_ch, y_ch, fn_ch = load_domain(PATHS_CH['AFIB'], PATHS_CH['SR'])
print(f'  CH total  : {len(X_ch):5d}  AFIB={y_ch.sum():4d}  SR={(y_ch==0).sum():4d}')

print('\nCarregando PTB-XL...')
X_ptb, y_ptb, fn_ptb = load_domain(PATHS_PTB['AFIB'], PATHS_PTB['SR'])
print(f'  PTB total  : {len(X_ptb):5d}  AFIB={y_ptb.sum():4d}  SR={(y_ptb==0).sum():4d}')

# ── Concatena no pool combinado ───────────────────────────────────────────
X_all  = np.concatenate([X_ch,  X_ptb])
y_all  = np.concatenate([y_ch,  y_ptb])
fn_all = np.concatenate([fn_ch, fn_ptb])
dom_all = np.concatenate([
    np.zeros(len(y_ch),  dtype=np.int64),   # 0 = CH
    np.ones( len(y_ptb), dtype=np.int64),   # 1 = PTB
])
print(f'\nPool total combinado: {len(y_all):5d}  '
      f'AFIB={y_all.sum():4d}  SR={(y_all==0).sum():4d}')

# ── Constrói vetor de patient_id por registro ────────────────────────────
print('\nConstruindo grupos patient-aware...')
groups_all = build_patient_groups(fn_all, dom_all)

# ── 1ª divisão: separa o holdout (~14,3%) preservando paciente ───────────
print('\n1ª divisão: StratifiedGroupKFold (holdout patient-aware)...')
sgkf_ho = StratifiedGroupKFold(
    n_splits=N_SPLITS_HOLDOUT, shuffle=True, random_state=SEED)
idx_trainval, idx_ho = select_holdout_fold(
    sgkf_ho, X_all, y_all, groups_all)

# ── 2ª divisão: separa calibração (≈5,0% do pool) preservando paciente ───────────
print('\n2ª divisão: StratifiedGroupKFold (calibração patient-aware)...')
sgkf_cal = StratifiedGroupKFold(
    n_splits=N_SPLITS_CALIB, shuffle=True, random_state=SEED)
idx_train_rel, idx_calib_rel = select_holdout_fold(
    sgkf_cal,
    X_all[idx_trainval], y_all[idx_trainval], groups_all[idx_trainval])
idx_train = idx_trainval[idx_train_rel]
idx_calib = idx_trainval[idx_calib_rel]

# ── Monta os três conjuntos ───────────────────────────────────────────────
X_train_raw = X_all[idx_train]
y_train     = y_all[idx_train]
dom_train   = dom_all[idx_train]
fn_train    = fn_all[idx_train]
groups_train = groups_all[idx_train]         # <— usado no 10-fold CV interno

X_calib_raw = X_all[idx_calib]
y_calib     = y_all[idx_calib]
dom_calib   = dom_all[idx_calib]

X_ho_raw    = X_all[idx_ho]
y_ho        = y_all[idx_ho]
dom_ho      = dom_all[idx_ho]
fn_ho       = fn_all[idx_ho]

# ── Salva índices + grupos para rastreabilidade e auditoria futura ───────
with open(CALIB_IDX_PATH, 'wb') as f:
    pickle.dump({
        'idx_train': idx_train, 'idx_calib': idx_calib, 'idx_ho': idx_ho,
        'groups_all': groups_all,   # NOVO: para verificação patient-level ex post
    }, f)

# ── Máscaras por domínio no holdout ──────────────────────────────────────
mask_ch  = dom_ho == 0
mask_ptb = dom_ho == 1

print(f'\n{"─"*60}')
print(f'  Split três vias patient-aware:')
print(f'  Treino  : {len(y_train):5d}  '
      f'AFIB={y_train.sum():4d}  SR={(y_train==0).sum():4d}  '
      f'CH={(dom_train==0).sum():4d}  PTB={(dom_train==1).sum():4d}')
print(f'  Calibr. : {len(y_calib):5d}  '
      f'AFIB={y_calib.sum():4d}  SR={(y_calib==0).sum():4d}  '
      f'CH={(dom_calib==0).sum():4d}  PTB={(dom_calib==1).sum():4d}')
print(f'  Holdout : {len(y_ho):5d}  '
      f'AFIB={y_ho.sum():4d}  SR={(y_ho==0).sum():4d}  '
      f'CH={mask_ch.sum():4d}  PTB={mask_ptb.sum():4d}')
print(f'  Ratio AFIB:SR no holdout: 1:{(y_ho==0).sum()/max(y_ho.sum(),1):.1f}')
print(f'{"─"*60}')

# ── Verificações patient-level (obrigatórias) ────────────────────────────
g_train = set(groups_all[idx_train])
g_calib = set(groups_all[idx_calib])
g_ho    = set(groups_all[idx_ho])
assert g_train.isdisjoint(g_ho),    'ERRO: paciente em treino e holdout.'
assert g_train.isdisjoint(g_calib), 'ERRO: paciente em treino e calibração.'
assert g_calib.isdisjoint(g_ho),    'ERRO: paciente em calibração e holdout.'
print('  ✅ Verificação patient-level: as três partições são disjuntas por paciente.')

# ── Verificação flexível de proporções (informativa, não bloqueante) ─────
frac_ho = len(y_ho) / len(y_all)
frac_afib_ho = y_ho.sum() / len(y_ho) if len(y_ho) > 0 else 0
print(f'  Fração holdout      : {frac_ho:.4f} (alvo ~0.15)')
print(f'  Fração AFIB holdout : {frac_afib_ho:.4f} (esperado ~0.117)')
assert 0.12 <= frac_ho <= 0.18, f'Fração holdout fora do intervalo aceitável: {frac_ho:.4f}'

# ── Planilha do holdout patient-aware ───────────────────────────────────────────
holdout_df = pd.DataFrame({
    'index_pool':  list(idx_ho),
    'filename':    [os.path.basename(f) for f in fn_ho],
    'filepath':    list(fn_ho),
    'label':       [{0:'SR', 1:'AFIB'}[int(l)] for l in y_ho],
    'dataset':     ['CH' if d==0 else 'PTB' for d in dom_ho],
    'label_int':   list(y_ho),
    'patient_group': list(groups_all[idx_ho]),
})
holdout_df.to_csv(HOLDOUT_CSV_PATH, index=False)
print(f'\nPlanilha holdout patient-aware salva: {HOLDOUT_CSV_PATH}')
print(f'  Linhas: {len(holdout_df)}   Colunas: {list(holdout_df.columns)}')
print(holdout_df.groupby(['dataset', 'label']).size().rename('n')
      .reset_index().to_string(index=False))

Carregando CH...
  Loading  [1780/1780] OK   
  Loading  [8125/8125] OK   
  CH total  :  9905  AFIB=1780  SR=8125

Carregando PTB-XL...
  Loading  [1514/1514] OK   
  Loading  [16748/16748] OK   
  PTB total  : 18262  AFIB=1514  SR=16748

Pool total combinado: 28167  AFIB=3294  SR=24873

Construindo grupos patient-aware...
  Grupos únicos: 26061  (para 28167 registros)

1ª divisão: StratifiedGroupKFold (holdout patient-aware)...
  Fold escolhida: 0  (fração AFIB holdout=0.1170, global=0.1169)

2ª divisão: StratifiedGroupKFold (calibração patient-aware)...
  Fold escolhida: 0  (fração AFIB holdout=0.1169, global=0.1169)

────────────────────────────────────────────────────────────
  Split três vias patient-aware:
  Treino  : 22722  AFIB=2657  SR=20065  CH=7939  PTB=14783
  Calibr. :  1420  AFIB= 166  SR=1254  CH= 501  PTB= 919
  Holdout :  4025  AFIB= 471  SR=3554  CH=1465  PTB=2560
  Ratio AFIB:SR no holdout: 1:7.5
────────────────────────────────────────────────────────────
  ✅ Verif

## 4. Extracao ACF e normalizacao

In [4]:
# X_train_raw, X_calib_raw e X_ho_raw contêm sinais filtrados com Butterworth.
# Ablation: SEM LOESS+NLM — apenas Butterworth (0.5–40 Hz).
# A extração de ACF e normalização seguem o fluxo original inalterado.

print('Extraindo ACF — pool de treino (pode levar 10-20 min)...')
X_train_acf_raw = extract_acf_batch(
    X_train_raw.reshape(-1, N_POINTS, 1), 'Treino')

print('Extraindo ACF — conjunto de calibracao INT8...')
X_calib_acf_raw = extract_acf_batch(
    X_calib_raw.reshape(-1, N_POINTS, 1), 'Calibracao')

print('Extraindo ACF — holdout...')
X_ho_acf_raw = extract_acf_batch(
    X_ho_raw.reshape(-1, N_POINTS, 1), 'Holdout')

# Scaler — fit APENAS no pool de treino
scaler = StandardScaler().fit(X_train_acf_raw)
with open(SCALER_PATH, 'wb') as f: pickle.dump(scaler, f)

X_train_acf = scaler.transform(X_train_acf_raw).astype(np.float32)
X_calib_acf = scaler.transform(X_calib_acf_raw).astype(np.float32)
X_ho_acf    = scaler.transform(X_ho_acf_raw).astype(np.float32)

# Reshape raw para (N, N_POINTS, 1)
X_train_raw = X_train_raw.reshape(-1, N_POINTS, 1)
X_calib_raw = X_calib_raw.reshape(-1, N_POINTS, 1)
X_ho_raw    = X_ho_raw.reshape(-1, N_POINTS, 1)

print(f'\nScaler salvo  : {SCALER_PATH}')
print(f'X_train_raw   : {X_train_raw.shape}')
print(f'X_train_acf   : {X_train_acf.shape}')
print(f'X_calib_raw   : {X_calib_raw.shape}')
print(f'X_calib_acf   : {X_calib_acf.shape}')
print(f'X_ho_raw      : {X_ho_raw.shape}')
print(f'X_ho_acf      : {X_ho_acf.shape}')


Extraindo ACF — pool de treino (pode levar 10-20 min)...
  Treino [22722/22722] OK          
Extraindo ACF — conjunto de calibracao INT8...
  Calibracao [1420/1420] OK          
Extraindo ACF — holdout...
  Holdout [4025/4025] OK          

Scaler salvo  : /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_scaler.pkl
X_train_raw   : (22722, 5000, 1)
X_train_acf   : (22722, 5000)
X_calib_raw   : (1420, 5000, 1)
X_calib_acf   : (1420, 5000)
X_ho_raw      : (4025, 5000, 1)
X_ho_acf      : (4025, 5000)


## 4b. Visualizacao de ondas — bruto vs filtrado (SEM LOESS+NLM)

Dois exemplos de AFIB e dois de SR selecionados aleatoriamente do pool de treino.
Cada linha mostra o sinal **bruto** (esquerda) e o sinal **apos o pipeline**
Butterworth (HP 0.5 Hz / LP 40 Hz) apenas (direita), nos primeiros 10 s do registro.

Os sinais brutos sao relidos do CSV original (sem filtrar) para comparacao direta.


In [5]:
# ── Selecao de exemplos: 2 AFIB + 2 SR do pool de treino ────────────────
np.random.seed(SEED + 7)   # semente distinta para variedade visual
idx_afib_tr = np.where(y_train == 1)[0]
idx_sr_tr   = np.where(y_train == 0)[0]
sel_afib = np.random.choice(idx_afib_tr, 2, replace=False)
sel_sr   = np.random.choice(idx_sr_tr,   2, replace=False)

def _load_raw_csv(filepath):
    """Le o sinal bruto (sem filtrar) do CSV original."""
    s = pd.read_csv(filepath, header=None).values.flatten().astype(np.float32)
    return s[:N_POINTS] if len(s) >= N_POINTS else np.pad(s, (0, N_POINTS - len(s)))

# Lista de (titulo, label, sinal_bruto, sinal_filtrado)
wave_examples = []
for i, (pool_idx, lbl) in enumerate([(sel_afib[0],'AFIB'),(sel_afib[1],'AFIB'),
                                      (sel_sr[0],  'SR'),  (sel_sr[1],  'SR')]):
    fp         = fn_train[pool_idx]
    raw_sig    = _load_raw_csv(fp)
    filt_sig   = X_train_raw[pool_idx].flatten()  # ja filtrado em load_domain()
    dom_str    = 'CH' if dom_train[pool_idx] == 0 else 'PTB'
    ex_num     = i % 2 + 1
    title      = f'{lbl} {ex_num} ({dom_str}) | {os.path.basename(fp)}'
    wave_examples.append((title, lbl, raw_sig, filt_sig))

# ── Figura: 4 linhas x 2 colunas ─────────────────────────────────────────
t_ax   = np.arange(N_POINTS) / FS
SHOW_N = min(int(10 * FS), N_POINTS)   # 10 segundos

fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
fig.suptitle(
    'ECGHybridV15 — Sinal bruto vs filtrado (apenas Butterworth, sem LOESS+NLM)',
    fontsize=11, fontweight='bold')

col_bruto   = '#7A7A7A'
col_labels  = {0: C_SR, 1: C_AFIB}   # SR=azul, AFIB=laranja

for row, (title, lbl, raw, filt) in enumerate(wave_examples):
    color = C_AFIB if lbl == 'AFIB' else C_SR

    ax0 = axes[row, 0]
    ax0.plot(t_ax[:SHOW_N], raw[:SHOW_N], color=col_bruto, lw=0.55, alpha=0.85)
    ax0.set_title(f'BRUTO  |  {title}', fontsize=7.5, loc='left', pad=3)
    ax0.set_ylabel('Amp. (mV)', fontsize=7)
    ax0.tick_params(labelsize=6.5)
    ax0.grid(True, alpha=0.18)
    for sp in ax0.spines.values(): sp.set_linewidth(1.2)
    ax0.spines['left'].set_color(color); ax0.spines['bottom'].set_color(color)

    ax1 = axes[row, 1]
    ax1.plot(t_ax[:SHOW_N], filt[:SHOW_N], color=color, lw=0.65)
    ax1.set_title(f'FILTRADO  |  {title}', fontsize=7.5, loc='left', pad=3, color=color)
    ax1.set_ylabel('Amp. (mV)', fontsize=7)
    ax1.tick_params(labelsize=6.5)
    ax1.grid(True, alpha=0.18)
    for sp in ax1.spines.values(): sp.set_linewidth(1.2)
    ax1.spines['left'].set_color(color); ax1.spines['bottom'].set_color(color)

axes[-1, 0].set_xlabel('Tempo (s)', fontsize=8)
axes[-1, 1].set_xlabel('Tempo (s)', fontsize=8)

from matplotlib.patches import Patch
legend_elems = [
    Patch(facecolor=col_bruto, label='Bruto'),
    Patch(facecolor=C_AFIB,   label='AFIB (filtrado)'),
    Patch(facecolor=C_SR,     label='SR (filtrado)'),
]
fig.legend(handles=legend_elems, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.01), fontsize=8.5)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(WAVE_PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot de ondas salvo: {WAVE_PLOT_PATH}')

Plot de ondas salvo: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_waves.png


/tmp/ipykernel_190719/2653930210.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Arquitetura — ECGHybridV15 com encode()

In [6]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, ks=3):
        super().__init__()
        p = ks // 2
        self.conv1 = nn.Conv1d(in_ch,  out_ch, ks, padding=p, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, ks, padding=p, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = (nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False),
                                    nn.BatchNorm1d(out_ch))
                      if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        return F.relu(
            self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.skip(x))


class ECGHybridV15(nn.Module):
    """v15 — sem CORAL (ablation sem domain adaptation)."""
    def __init__(self):
        super().__init__()
        d = 128
        self.stem     = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4))
        self.res1     = ResidualBlock1D(32, 32, 5); self.pool1 = nn.MaxPool1d(4)
        self.res2     = ResidualBlock1D(32, 64, 3); self.pool2 = nn.MaxPool1d(4)
        self.res3     = ResidualBlock1D(64, d,  3)
        self.cnn_head = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.3))
        self.acf_conv = nn.Sequential(
            nn.Conv1d(1,  16, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=4, padding=2, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(16), nn.ReLU())
        self.acf_head   = nn.Sequential(nn.Linear(16, 16), nn.ReLU())
        self.gate       = nn.Sequential(nn.Linear(80, 80), nn.Sigmoid())
        self.classifier = nn.Sequential(
            nn.Linear(80, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))

    def forward(self, x_raw, x_acf):
        x = self.stem(x_raw.permute(0, 2, 1))
        x = self.pool1(self.res1(x)); x = self.pool2(self.res2(x)); x = self.res3(x)
        x = self.cnn_head(x.mean(dim=-1))
        a = self.acf_head(self.acf_conv(x_acf.unsqueeze(1)).mean(dim=-1))
        c = torch.cat([x, a], dim=1)
        feat = c * self.gate(c)
        return torch.sigmoid(self.classifier(feat)).squeeze(-1)


def focal_loss(pred, target, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    pred_c = torch.clamp(pred, 1e-7, 1-1e-7)
    bce    = -target*torch.log(pred_c) - (1-target)*torch.log(1-pred_c)
    p_t    = target*pred_c + (1-target)*(1-pred_c)
    w      = target*alpha  + (1-target)*(1-alpha)
    return (w*(1-p_t)**gamma*bce).mean()


n = sum(p.numel() for p in ECGHybridV15().parameters())
print(f'ECGHybridV15 (SEM CORAL — ablation): {n:,} parametros')

ECGHybridV15 (SEM CORAL — ablation): 138,801 parametros


## 6. (Ablation) CORAL removida + SEM LOESS+NLM

Nesta variante, a **Deep CORAL Loss** foi removida por completo.  
O treino utiliza apenas a **Focal Loss**, sem alinhamento de domínio explícito.  
Adicionalmente, **Ablation: LOESS e NLM removidos (apenas Butterworth)**.  
Arquitetura idêntica à v15 original — somente a função de perda e o pipeline de filtros diferem.


In [7]:
# CoralLoss REMOVIDA nesta ablation.
# O treino usa apenas focal_loss definida na célula anterior.
print('Ablation sem CORAL: apenas Focal Loss no treino.')

Ablation sem CORAL: apenas Focal Loss no treino.


## 7. Dataset (sem rótulo de domínio para CORAL)

O `domain` ainda é carregado para permitir avaliação separada por domínio no holdout,  
mas **não é usado na função de perda**.

In [8]:
class ECGDataset(Dataset):
    """Dataset sem rótulo de domínio (ablation sem CORAL)."""
    def __init__(self, X_raw, X_feat, y, augment=False):
        self.X_raw  = torch.tensor(X_raw,  dtype=torch.float32)
        self.X_feat = torch.tensor(X_feat, dtype=torch.float32)
        self.y      = torch.tensor(y,      dtype=torch.float32)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        xr, xf = self.X_raw[idx].clone(), self.X_feat[idx].clone()
        if self.augment and random.random() < AUG_PROB:
            snr  = random.uniform(AUG_SNR_MIN, AUG_SNR_MAX)
            nvar = xr.pow(2).mean() / (10 ** (snr/10))
            xr   = xr + torch.randn_like(xr) * nvar.sqrt()
            xr   = xr * (1 + random.uniform(-AUG_AMP_RANGE, AUG_AMP_RANGE))
        return xr, xf, self.y[idx]

print('ECGDataset (sem domínio) definido.')

ECGDataset (sem domínio) definido.


## 8. Funcoes de avaliacao

In [9]:
def eval_ds(model, X_raw, X_acf, y):
    """Inferencia em um dataset. Retorna array de probabilidades."""
    ds = ECGDataset(X_raw, X_acf, y, augment=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                    worker_init_fn=seed_worker)
    probs = []
    with torch.no_grad():
        for xr, xf, _ in dl:
            probs += model(xr.to(DEVICE), xf.to(DEVICE)).cpu().tolist()
    return np.array(probs)


def metrics(probs, labels, tau):
    """Metricas de classificacao dado threshold tau."""
    preds = (probs >= tau).astype(int)
    tp = ((preds==1)&(labels==1)).sum(); tn = ((preds==0)&(labels==0)).sum()
    fp = ((preds==1)&(labels==0)).sum(); fn = ((preds==0)&(labels==1)).sum()
    return dict(
        auc  = roc_auc_score(labels, probs),
        sens = tp/(tp+fn+1e-8),
        spec = tn/(tn+fp+1e-8),
        ppv  = tp/(tp+fp+1e-8),
        f1   = 2*tp/(2*tp+fp+fn+1e-8),
        tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn))

print('Funcoes de avaliacao definidas.')

Funcoes de avaliacao definidas.


## 9. Loop de treino — 10-Fold CV (sem CORAL)

Idêntico ao original, **exceto** que não há `coral_criterion` nem `LAMBDA_CORAL`.  
Cada batch calcula apenas a Focal Loss.

In [10]:
# Patient-aware CV: StratifiedGroupKFold garante que nenhum paciente cruze folds
kf           = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []
best_val_auc = -1; best_fold = -1

for fold, (tr_idx, val_idx) in enumerate(
        kf.split(X_train_raw, y_train, groups=groups_train), 1):
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'─'*60}")

    X_tr_r = X_train_raw[tr_idx]; X_val_r = X_train_raw[val_idx]
    X_tr_f = X_train_acf[tr_idx]; X_val_f = X_train_acf[val_idx]
    y_tr   = y_train[tr_idx];     y_val   = y_train[val_idx]
    dom_tr = dom_train[tr_idx];   dom_val = dom_train[val_idx]

    print(f"  Treino: {len(y_tr):5d}  "
          f"CH={(dom_tr==0).sum():4d}  PTB={(dom_tr==1).sum():4d}  "
          f"AFIB={y_tr.sum():4d}")
    print(f"  Val   : {len(y_val):5d}  "
          f"CH={(dom_val==0).sum():4d}  PTB={(dom_val==1).sum():4d}  "
          f"AFIB={y_val.sum():4d}")

    set_seed(SEED + fold)                              # v3: fold reprodutível isoladamente
    g_fold = torch.Generator(); g_fold.manual_seed(SEED + fold)
    tr_ds  = ECGDataset(X_tr_r,  X_tr_f,  y_tr,  augment=True)
    val_ds = ECGDataset(X_val_r, X_val_f, y_val, augment=False)

    w       = 1.0 / np.bincount(y_tr)[y_tr]
    sampler = WeightedRandomSampler(
        torch.from_numpy(w).float(), len(w), replacement=True, generator=g_fold)
    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, sampler=sampler,
                            num_workers=2, pin_memory=True,
                            worker_init_fn=seed_worker, generator=g_fold)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True,
                            worker_init_fn=seed_worker)

    model = ECGHybridV15().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR_INITIAL, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=1e-5)

    best_auc_fold = -1; patience_cnt = 0; best_state = None
    history = {'focal': [], 'val_auc': []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_focal = 0.0; n_bat = 0

        for xr, xf, yb in tr_loader:
            xr = xr.to(DEVICE); xf = xf.to(DEVICE)
            yb = yb.float().to(DEVICE)
            opt.zero_grad()

            pred = model(xr, xf)
            l_focal = focal_loss(pred, yb)
            l_focal.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_focal += l_focal.item(); n_bat += 1

        sched.step()
        model.eval()
        probs, labels = [], []

        with torch.no_grad():
            for xr, xf, yb in val_loader:   # <-- era: for xr, xf, yb, _ in val_loader:
                probs  += model(xr.to(DEVICE), xf.to(DEVICE)).cpu().tolist()
                labels += yb.tolist()
        val_auc = roc_auc_score(labels, probs)

        avg_f = ep_focal / n_bat
        history['focal'].append(avg_f)
        history['val_auc'].append(val_auc)

        if val_auc > best_auc_fold:
            best_auc_fold = val_auc; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1

        if epoch % 10 == 0:
            print(f"  Ep {epoch:3d} | AUC={val_auc:.4f} best={best_auc_fold:.4f} "
                  f"L_f={avg_f:.4f}")
        if patience_cnt >= PATIENCE:
            print(f"  Early stop ep {epoch} | best={best_auc_fold:.4f}")
            break

    # ── Metricas do fold ──────────────────────────────────────────────────
    model.load_state_dict(best_state); model.eval()

    # Threshold otimo (Youden Index = Sens + Spec - 1) na particao de validacao do fold
    p_val = eval_ds(model, X_val_r, X_val_f, y_val)
    fpr, tpr, roc_thr = roc_curve(y_val, p_val)
    youden = tpr - fpr                             # J = Sens + Spec - 1
    opt_thresh = float(roc_thr[np.argmax(youden)])
    m_val = metrics(p_val, y_val, opt_thresh)

    print(f"\n  Fold {fold} | tau={opt_thresh:.4f} | "
          f"Val AUC={m_val['auc']:.4f} Sens={m_val['sens']:.4f} "
          f"Spec={m_val['spec']:.4f} F1={m_val['f1']:.4f}")

    fold_results.append(dict(
        fold=fold, val_auc=best_auc_fold,
        threshold=opt_thresh, history=history,
        m_val=m_val, state=best_state))

    if best_auc_fold > best_val_auc:
        best_val_auc = best_auc_fold; best_fold = fold
        # ── Guardar dados de validacao do melhor fold ─────────────────
        # Serao usados para recalibrar o threshold no modelo TFLite INT8
        best_fold_val_raw = X_val_r.copy()
        best_fold_val_acf = X_val_f.copy()
        best_fold_val_y   = y_val.copy()
        print(f"    -> Dados de validacao do fold {fold} salvos para "
              f"calibracao INT8 (n={len(y_val)})")

print(f"\n{'='*60}")
print(f"  {N_FOLDS}-FOLD — SEM CORAL + SEM LOESS+NLM (ablation)")
print(f"{'='*60}")
for label, k in [('Val AUC', 'auc'), ('Val Sens', 'sens'),
                  ('Val Spec', 'spec'), ('Val F1', 'f1')]:
    v = np.array([r['m_val'][k] for r in fold_results])
    print(f"  {label:<12}: {v.mean():.4f} +/- {v.std():.4f} "
          f"[{v.min():.4f} - {v.max():.4f}]")


────────────────────────────────────────────────────────────
  FOLD 1/10
────────────────────────────────────────────────────────────
  Treino: 20450  CH=7158  PTB=13292  AFIB=2391
  Val   :  2272  CH= 781  PTB=1491  AFIB= 266
  Ep  10 | AUC=0.9892 best=0.9950 L_f=0.0078
  Ep  20 | AUC=0.9952 best=0.9952 L_f=0.0063
  Ep  30 | AUC=0.9933 best=0.9955 L_f=0.0058
  Ep  40 | AUC=0.9894 best=0.9960 L_f=0.0039
  Ep  50 | AUC=0.9941 best=0.9960 L_f=0.0034
  Ep  60 | AUC=0.9942 best=0.9963 L_f=0.0025
  Early stop ep 66 | best=0.9963

  Fold 1 | tau=0.6042 | Val AUC=0.9963 Sens=0.9774 Spec=0.9870 F1=0.9420
    -> Dados de validacao do fold 1 salvos para calibracao INT8 (n=2272)

────────────────────────────────────────────────────────────
  FOLD 2/10
────────────────────────────────────────────────────────────
  Treino: 20450  CH=7152  PTB=13298  AFIB=2391
  Val   :  2272  CH= 787  PTB=1485  AFIB= 266
  Ep  10 | AUC=0.9981 best=0.9986 L_f=0.0077
  Ep  20 | AUC=0.9988 best=0.9988 L_f=0.0066
  Ep

## 10. Selecao do modelo final e avaliacao no holdout

Apenas aqui o holdout e utilizado, uma unica vez.

In [11]:
# ── Selecao pelo melhor val_AUC do K-Fold ────────────────────────────────
print(f"{'='*62}")
print(f"  SELECAO — criterio: val_AUC K-Fold")
print(f"{'='*62}")
print(f"{'Fold':>4} {'val_AUC':>9} {'tau':>8}")
print('─'*28)
for r in fold_results:
    mk = ' <-' if r['fold'] == best_fold else ''
    print(f"  {r['fold']:>2}  {r['val_auc']:>9.4f}  {r['threshold']:>8.4f}{mk}")

best       = fold_results[best_fold - 1]
best_model = ECGHybridV15()
best_model.load_state_dict(best['state'])
best_model.eval().to(DEVICE)
torch.save(best['state'], FINAL_MODEL_PATH)
THRESHOLD_FINAL = best['threshold']
with open(THRESH_PATH, 'wb') as f: pickle.dump(THRESHOLD_FINAL, f)
print(f"\nModelo salvo: {FINAL_MODEL_PATH}")
print(f"tau float32 (Youden): {THRESHOLD_FINAL:.6f}")
print(f"\nDados de validacao do best fold {best_fold} salvos:")
print(f"  best_fold_val_raw : {best_fold_val_raw.shape}")
print(f"  best_fold_val_acf : {best_fold_val_acf.shape}")
print(f"  best_fold_val_y   : {best_fold_val_y.shape}  "
      f"(AFIB={best_fold_val_y.sum():.0f}, SR={(1-best_fold_val_y).sum():.0f})")

# ── Avaliacao no holdout (primeira e unica vez) ───────────────────────────
print(f"\n{'='*62}")
print(f"  AVALIACAO NO HOLDOUT PATIENT-AWARE (≈14,3% do pool — pacientes nunca vistos no treino)")
print(f"{'='*62}")

p_ho_all = eval_ds(best_model, X_ho_raw, X_ho_acf, y_ho)

# Total
m_ho_total = metrics(p_ho_all, y_ho, THRESHOLD_FINAL)

# Por dominio
mask_ch  = dom_ho == 0
mask_ptb = dom_ho == 1
m_ho_ch  = metrics(p_ho_all[mask_ch],  y_ho[mask_ch],  THRESHOLD_FINAL)
m_ho_ptb = metrics(p_ho_all[mask_ptb], y_ho[mask_ptb], THRESHOLD_FINAL)

def print_metrics(label, m):
    print(f"  {label:<18} AUC={m['auc']:.4f}  Sens={m['sens']:.4f}  "
          f"Spec={m['spec']:.4f}  PPV={m['ppv']:.4f}  F1={m['f1']:.4f}")
    print(f"  {'':18} TP={m['tp']}  TN={m['tn']}  FP={m['fp']}  FN={m['fn']}")

print_metrics('TOTAL (CH+PTB)', m_ho_total)
print()
print_metrics('CH  holdout', m_ho_ch)
print()
print_metrics('PTB holdout', m_ho_ptb)

  SELECAO — criterio: val_AUC K-Fold
Fold   val_AUC      tau
────────────────────────────
   1     0.9963    0.6042
   2     0.9991    0.7916 <-
   3     0.9954    0.6340
   4     0.9966    0.7196
   5     0.9987    0.6196
   6     0.9981    0.4619
   7     0.9978    0.6849
   8     0.9971    0.6598
   9     0.9962    0.6653
  10     0.9971    0.7300

Modelo salvo: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_final.pt
tau float32 (Youden): 0.791590

Dados de validacao do best fold 2 salvos:
  best_fold_val_raw : (2272, 5000, 1)
  best_fold_val_acf : (2272, 5000)
  best_fold_val_y   : (2272,)  (AFIB=266, SR=2006)

  AVALIACAO NO HOLDOUT PATIENT-AWARE (≈14,3% do pool — pacientes nunca vistos no treino)
  TOTAL (CH+PTB)     AUC=0.9971  Sens=0.9703  Spec=0.9868  PPV=0.9067  F1=0.9374
                     TP=457  TN=3507  FP=47  FN=14

  CH  holdout        AUC=0.9984  Sens=0.9770  Spec=0.9867  PPV=0.9410  F1=0.9586
                     TP=255  TN=

## 11. Diagnóstico do treino (sem CORAL)

In [12]:
h = best['history']
ep = range(1, len(h['focal']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(
    f'{VARIANT_LABEL} — Fold {best_fold}',
    fontsize=12, y=1.01)

axes[0].plot(ep, h['focal'], color=C_SR, lw=1.5)
axes[0].set_title('Focal Loss (treino)'); axes[0].set_xlabel('Epoca')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, h['val_auc'], color='#1D9E75', lw=1.5)
axes[1].axhline(max(h['val_auc']), color='#1D9E75', ls='--', lw=0.8, alpha=0.6)
axes[1].set_title(f"Val AUC K-Fold (max={max(h['val_auc']):.4f})")
axes[1].set_xlabel('Epoca'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figura salva: {RESULTS_PATH}')

Figura salva: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_results.png


## 12. Curva ROC float32 — holdout patient-aware (CH + PTB)

Curva ROC do modelo float32 (PyTorch) sobre o holdout patient-aware, separada por
domínio e agregada. A ROC INT8 (referência do firmware) é gerada na última célula.

In [13]:
def roc(p, l):
    f, t, _ = roc_curve(l, p)
    return f, t, sk_auc(f, t)

fpr_ch,  tpr_ch,  auc_ch  = roc(p_ho_all[mask_ch],  y_ho[mask_ch])
fpr_ptb, tpr_ptb, auc_ptb = roc(p_ho_all[mask_ptb], y_ho[mask_ptb])
fpr_tot, tpr_tot, auc_tot = roc(p_ho_all,            y_ho)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(fpr_tot, tpr_tot, color='#1D9E75', lw=2.2, ls='--',
        label=f'Total holdout   AUC = {auc_tot:.4f}')
ax.plot(fpr_ch,  tpr_ch,  color=C_SR,   lw=2.0,
        label=f'CH  holdout     AUC = {auc_ch:.4f}')
ax.plot(fpr_ptb, tpr_ptb, color=C_AFIB, lw=2.0,
        label=f'PTB holdout     AUC = {auc_ptb:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.4, label='Aleatorio')

# Pontos de operacao (tau)
ax.scatter([1 - m_ho_ch['spec']],  [m_ho_ch['sens']],
           color=C_SR,   s=90, zorder=6, edgecolors='white', linewidths=0.8)
ax.scatter([1 - m_ho_ptb['spec']], [m_ho_ptb['sens']],
           color=C_AFIB, s=90, zorder=6, edgecolors='white', linewidths=0.8)
ax.scatter([1 - m_ho_total['spec']], [m_ho_total['sens']],
           color='#1D9E75', s=90, zorder=6, marker='D',
           edgecolors='white', linewidths=0.8,
           label=f'tau = {THRESHOLD_FINAL:.4f}')

# Anotacoes dos pontos
for (spec, sens, label_txt, clr) in [
        (m_ho_ch['spec'],    m_ho_ch['sens'],    'CH',    C_SR),
        (m_ho_ptb['spec'],   m_ho_ptb['sens'],   'PTB',   C_AFIB),
        (m_ho_total['spec'], m_ho_total['sens'], 'Total', '#1D9E75')]:
    ax.annotate(
        f'{label_txt}\nSens={sens:.3f}\nSpec={spec:.3f}',
        xy=(1 - spec, sens), xytext=(8, -18),
        textcoords='offset points', fontsize=7,
        color=clr, arrowprops=dict(arrowstyle='->', color=clr, lw=0.8))

ax.set_xlabel('FPR  (1 - Especificidade)', fontsize=10)
ax.set_ylabel('TPR  (Sensibilidade)',       fontsize=10)
ax.set_title(
    f'Curva ROC (float32) — {VARIANT_LABEL}\n'
    f'Holdout patient-aware: CH + PTB  (n={len(y_ho)})',
    fontsize=10)
ax.legend(fontsize=8.5, loc='lower right')
ax.grid(True, alpha=0.22)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig(ROC_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curva ROC salva: {ROC_PATH}')
print(f'  AUC total : {auc_tot:.4f}')
print(f'  AUC CH    : {auc_ch:.4f}')
print(f'  AUC PTB   : {auc_ptb:.4f}')

Curva ROC salva: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_roc.png
  AUC total : 0.9971
  AUC CH    : 0.9984
  AUC PTB   : 0.9960


/tmp/ipykernel_190719/1681778291.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Grid search de lambda (recomendado antes do 10-fold)

Criterio: AUC no holdout PTB-XL (dominio com maior domain shift).  
Tempo estimado: ~15 min por lambda na RTX 5060 Ti.

In [14]:
# Descomente para executar
#
# LAMBDA_CANDIDATES = [0.1, 0.5, 1.0, 2.0]
# QUICK_EPOCHS = 30
# grid_results = {}
#
# # Usa apenas o primeiro fold para speed
# tr_idx_gs, val_idx_gs = next(
#     StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_train_raw, y_train))
#
# for lam in LAMBDA_CANDIDATES:
#     print(f'\nlambda = {lam}')
#     m_gs   = ECGHybridV15().to(DEVICE)
#     opt_gs = torch.optim.Adam(m_gs.parameters(), lr=LR_INITIAL, weight_decay=1e-4)
#     sched_gs = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gs, QUICK_EPOCHS, 1e-5)
#     ds_gs = ECGDataset(
#         X_train_raw[tr_idx_gs], X_train_acf[tr_idx_gs],
#         y_train[tr_idx_gs], dom_train[tr_idx_gs], augment=True)
#     w_gs  = 1.0 / np.bincount(y_train[tr_idx_gs])[y_train[tr_idx_gs]]
#     ldr_gs = DataLoader(ds_gs, batch_size=BATCH_SIZE,
#                         sampler=WeightedRandomSampler(
#                             torch.from_numpy(w_gs).float(), len(w_gs)),
#                         num_workers=2, pin_memory=True)
#     for ep in range(1, QUICK_EPOCHS+1):
#         m_gs.train()
#         for xr, xf, yb, dom in ldr_gs:
#             xr,xf,yb,dom = (xr.to(DEVICE), xf.to(DEVICE),
#                              yb.float().to(DEVICE), dom.to(DEVICE))
#             opt_gs.zero_grad()
#             feat = m_gs.encode(xr, xf)
#             pred = torch.sigmoid(m_gs.classifier(feat)).squeeze(-1)
#             l_f  = focal_loss(pred, yb)
#             mc, mp = dom==0, dom==1
#             l_c = (coral_criterion(feat[mc], feat[mp])
#                    if mc.sum()>=2 and mp.sum()>=2
#                    else torch.tensor(0.0, device=DEVICE))
#             (l_f + lam*l_c).backward()
#             torch.nn.utils.clip_grad_norm_(m_gs.parameters(), 1.0)
#             opt_gs.step()
#         sched_gs.step()
#     # Avalia no holdout PTB (nao foi usado no treino)
#     m_gs.eval()
#     p_ptb_gs = eval_ds(m_gs, X_ho_raw[mask_ptb], X_ho_acf[mask_ptb],
#                        y_ho[mask_ptb], dom_ho[mask_ptb])
#     auc_gs = roc_auc_score(y_ho[mask_ptb], p_ptb_gs)
#     grid_results[lam] = auc_gs
#     print(f'  PTB holdout AUC @ lambda={lam}: {auc_gs:.4f}')
#
# best_lam = max(grid_results, key=grid_results.get)
# print(f'\nMelhor lambda: {best_lam}  PTB AUC={grid_results[best_lam]:.4f}')
print('Grid search desabilitado. Descomente para usar.')

Grid search desabilitado. Descomente para usar.


## 14. Construção do modelo Keras

Replica a arquitetura v15 em TensorFlow/Keras para conversão TFLite INT8.  
`FixedBatchNorm` reproduz o BatchNorm1d PyTorch em modo eval (pesos fixos).

In [15]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
print(f'TensorFlow : {tf.__version__} (CPU)')


class FixedBatchNorm(tf.keras.layers.Layer):
    """Replica BatchNorm1d PyTorch em modo eval (pesos fixos)."""
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs); self.epsilon = epsilon
    def build(self, input_shape):
        c = input_shape[-1]
        self.gamma = self.add_weight(shape=(c,), initializer='ones',  trainable=True,  name='gamma')
        self.beta  = self.add_weight(shape=(c,), initializer='zeros', trainable=True,  name='beta')
        self.rm    = self.add_weight(shape=(c,), initializer='zeros', trainable=False, name='moving_mean')
        self.rv    = self.add_weight(shape=(c,), initializer='ones',  trainable=False, name='moving_variance')
    def call(self, x, training=None):
        return tf.nn.batch_normalization(x, self.rm, self.rv, self.beta, self.gamma, self.epsilon)
    def get_config(self):
        cfg = super().get_config(); cfg.update({'epsilon': self.epsilon}); return cfg


def res_block_tf(x, filters, ks, name):
    sc = x
    x  = tf.keras.layers.Conv1D(filters, ks, padding='same', use_bias=False, name=f'{name}_c1')(x)
    x  = FixedBatchNorm(name=f'{name}_bn1')(x)
    x  = tf.keras.layers.ReLU()(x)
    x  = tf.keras.layers.Conv1D(filters, ks, padding='same', use_bias=False, name=f'{name}_c2')(x)
    x  = FixedBatchNorm(name=f'{name}_bn2')(x)
    if sc.shape[-1] != filters:
        sc = tf.keras.layers.Conv1D(filters, 1, use_bias=False, name=f'{name}_skip_conv')(sc)
        sc = FixedBatchNorm(name=f'{name}_skip_bn')(sc)
    return tf.keras.layers.ReLU()(tf.keras.layers.Add()([x, sc]))


def build_keras_model_v15():
    """Arquitetura v15 identica ao original (ablation sem CORAL)."""
    inp_raw = tf.keras.Input(shape=(N_POINTS, 1),  name='input_raw')
    inp_acf = tf.keras.Input(shape=(N_ACF_LAGS,),  name='input_acf')

    x = tf.keras.layers.Conv1D(32, 7, padding='same', use_bias=False, name='stem_conv')(inp_raw)
    x = FixedBatchNorm(name='stem_bn')(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = res_block_tf(x, 32,  5, 'res1'); x = tf.keras.layers.MaxPooling1D(4)(x)
    x = res_block_tf(x, 64,  3, 'res2'); x = tf.keras.layers.MaxPooling1D(4)(x)
    x = res_block_tf(x, 128, 3, 'res3')
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation='relu', name='cnn_head')(x)

    a = tf.keras.layers.Reshape((N_ACF_LAGS, 1))(inp_acf)
    a = tf.keras.layers.ZeroPadding1D(padding=3)(a)
    a = tf.keras.layers.Conv1D(16, 7, strides=4, padding='valid', use_bias=False, name='acf_c1')(a)
    a = FixedBatchNorm(name='acf_bn1')(a); a = tf.keras.layers.ReLU()(a)
    a = tf.keras.layers.ZeroPadding1D(padding=2)(a)
    a = tf.keras.layers.Conv1D(32, 5, strides=4, padding='valid', use_bias=False, name='acf_c2')(a)
    a = FixedBatchNorm(name='acf_bn2')(a); a = tf.keras.layers.ReLU()(a)
    a = tf.keras.layers.ZeroPadding1D(padding=1)(a)
    a = tf.keras.layers.Conv1D(16, 3, strides=2, padding='valid', use_bias=False, name='acf_c3')(a)
    a = FixedBatchNorm(name='acf_bn3')(a); a = tf.keras.layers.ReLU()(a)
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    a = tf.keras.layers.Dense(16, activation='relu', name='acf_head')(a)

    c   = tf.keras.layers.Concatenate()([x, a])
    g   = tf.keras.layers.Dense(80, activation='sigmoid', name='gate')(c)
    c   = tf.keras.layers.Multiply()([c, g])
    c   = tf.keras.layers.Dense(64, activation='relu', name='clf1')(c)
    out = tf.keras.layers.Dense(1,  activation='sigmoid', name='clf2')(c)
    return tf.keras.Model(inputs=[inp_raw, inp_acf], outputs=out, name='ecg_hybrid_v15_sem_loess_nlm')


keras_model = build_keras_model_v15()
print(f'Modelo Keras v15 (sem CORAL): {keras_model.count_params():,} parametros')

E0000 00:00:1790529017.522767  190719 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790529017.526164  190719 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790529017.535846  190719 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790529017.535856  190719 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790529017.535858  190719 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790529017.535859  190719 computation_placer.cc:177] computation placer already registered. Please check linka

TensorFlow : 2.19.0 (CPU)
Modelo Keras v15 (sem CORAL): 140,273 parametros


W0000 00:00:1790529018.481707  190719 gpu_device.cc:2430] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 15. Copia de pesos PyTorch → Keras

In [16]:
src = torch.load(FINAL_MODEL_PATH, map_location='cpu')

def t(key): return src[key].numpy()
def set_conv1d(layer, wk): layer.set_weights([t(wk).transpose(2, 1, 0)])
def set_fbn(layer, p):
    layer.set_weights([t(f'{p}.weight'), t(f'{p}.bias'),
                       t(f'{p}.running_mean'), t(f'{p}.running_var')])
def set_fbn_safe_v2(layer, p):
    """Neutraliza canais mortos (running_var ~ 0) no ramo ACF."""
    gamma  = t(f'{p}.weight').copy(); beta   = t(f'{p}.bias').copy()
    r_mean = t(f'{p}.running_mean').copy(); r_var = t(f'{p}.running_var').copy()
    bad = r_var < 1e-10
    if bad.sum() > 0:
        print(f'    {p}: {bad.sum():2d}/{len(bad)} canais mortos neutralizados')
        r_mean[bad] = 0.0; r_var[bad] = 1.0; gamma[bad] = 0.0; beta[bad] = 0.0
    layer.set_weights([gamma, beta, r_mean, r_var])
def set_dense(layer, wk, bk): layer.set_weights([t(wk).T, t(bk)])
def set_resblock(kn, pn, has_skip=False):
    set_conv1d(keras_model.get_layer(f'{kn}_c1'), f'{pn}.conv1.weight')
    set_fbn(keras_model.get_layer(f'{kn}_bn1'),   f'{pn}.bn1')
    set_conv1d(keras_model.get_layer(f'{kn}_c2'), f'{pn}.conv2.weight')
    set_fbn(keras_model.get_layer(f'{kn}_bn2'),   f'{pn}.bn2')
    if has_skip:
        set_conv1d(keras_model.get_layer(f'{kn}_skip_conv'), f'{pn}.skip.0.weight')
        set_fbn(keras_model.get_layer(f'{kn}_skip_bn'),      f'{pn}.skip.1')

print('Copiando pesos PyTorch (SEM CORAL) -> Keras:')
print('  Ramo CNN...')
set_conv1d(keras_model.get_layer('stem_conv'), 'stem.0.weight')
set_fbn(keras_model.get_layer('stem_bn'), 'stem.1')
set_resblock('res1', 'res1', has_skip=False)
set_resblock('res2', 'res2', has_skip=True)
set_resblock('res3', 'res3', has_skip=True)
set_dense(keras_model.get_layer('cnn_head'), 'cnn_head.0.weight', 'cnn_head.0.bias')
print('    OK CNN')
print('  Ramo ACF...')
set_conv1d(keras_model.get_layer('acf_c1'), 'acf_conv.0.weight')
set_fbn_safe_v2(keras_model.get_layer('acf_bn1'), 'acf_conv.1')
set_conv1d(keras_model.get_layer('acf_c2'), 'acf_conv.3.weight')
set_fbn_safe_v2(keras_model.get_layer('acf_bn2'), 'acf_conv.4')
set_conv1d(keras_model.get_layer('acf_c3'), 'acf_conv.6.weight')
set_fbn_safe_v2(keras_model.get_layer('acf_bn3'), 'acf_conv.7')
set_dense(keras_model.get_layer('acf_head'), 'acf_head.0.weight', 'acf_head.0.bias')
print('    OK ACF')
print('  Fusao (gate + classificador)...')
set_dense(keras_model.get_layer('gate'), 'gate.0.weight',       'gate.0.bias')
set_dense(keras_model.get_layer('clf1'), 'classifier.0.weight', 'classifier.0.bias')
set_dense(keras_model.get_layer('clf2'), 'classifier.3.weight', 'classifier.3.bias')
print('    OK Fusao')

# ── Verificacao numerica PyTorch vs Keras ─────────────────────────────────
print('\nVerificacao numerica PyTorch vs Keras...')
m_pt = ECGHybridV15(); m_pt.load_state_dict(src); m_pt.eval()
torch.manual_seed(42)
xr_t = torch.randn(2, N_POINTS, 1); xacf_t = torch.randn(2, N_ACF_LAGS)
with torch.no_grad(): out_pt = m_pt(xr_t, xacf_t).numpy()
out_kf = keras_model([xr_t.numpy(), xacf_t.numpy()], training=False).numpy().flatten()
diff   = abs(out_pt.flatten() - out_kf).max()
print(f'  PyTorch : {out_pt.flatten()}')
print(f'  Keras   : {out_kf.flatten()}')
print(f'  Diff max: {diff:.2e}  {"OK" if diff < 5e-3 else "VERIFICAR"}')
assert diff < 5e-3, f'Diferenca muito alta: {diff:.2e}'

Copiando pesos PyTorch (SEM CORAL) -> Keras:
  Ramo CNN...
    OK CNN
  Ramo ACF...
    acf_conv.1:  8/16 canais mortos neutralizados
    acf_conv.4: 26/32 canais mortos neutralizados
    acf_conv.7:  5/16 canais mortos neutralizados
    OK ACF
  Fusao (gate + classificador)...
    OK Fusao

Verificacao numerica PyTorch vs Keras...
  PyTorch : [0.48066634 0.46930525]
  Keras   : [0.48066634 0.46930522]
  Diff max: 2.98e-08  OK


## 16. Conversão TFLite INT8 e header C

In [17]:
# ═════════════════════════════════════════════════════════════════════════
# CALIBRACAO E CONVERSAO TFLite INT8 (com winsorizacao anti-outlier)
# ═════════════════════════════════════════════════════════════════════════
# CORRECAO CRITICA v3: winsorizacao do representative dataset.
#
# O sinal ECG RAW pode conter outliers extremos (>=15 mV) causados por
# eletrodo desconectado ou artefato de movimento severo. Como o quantizador
# INT8 do TFLite usa o valor absoluto maximo do representative dataset para
# calcular os scales dos tensores, alguns poucos sinais anomalos (<0.5% do
# total) sao suficientes para inflar o scale de entrada em 4-5x, esmagando
# a resolucao efetiva do modelo. O sintoma e um Pearson r ~ 0.5 entre INT8
# e Keras Float32, com AUC caindo de 0.998 para 0.957.
#
# Solucao: aplicar clip nos sinais do representative dataset usando como
# limite os quantis empiricos p0.1 e p99.9 do proprio conjunto de treino,
# preservando 99.8% dos dados fisiologicos intactos e apenas descartando
# outliers de artefato. A ACF nao precisa de winsorizacao porque ja e
# normalizada pelo StandardScaler.

print('Preparando dados de calibracao INT8 (conjunto formal 5%)...')

# ── Winsorizacao do RAW baseada em quantis empiricos do treino ───────────
raw_p999 = np.percentile(X_train_raw,  99.9)
raw_p001 = np.percentile(X_train_raw,   0.1)
raw_clip = max(abs(raw_p999), abs(raw_p001))
print(f'  Clip anti-outlier RAW: +/-{raw_clip:.4f} mV')
print(f'    (baseado em p99.9={raw_p999:+.4f} / p0.1={raw_p001:+.4f} do treino)')

calib_raw_np = np.clip(X_calib_raw, -raw_clip, raw_clip).astype(np.float32)
calib_raw_np = calib_raw_np.reshape(-1, N_POINTS, 1)
calib_acf_np = X_calib_acf.astype(np.float32)   # ja normalizada pelo scaler

n_clipped_train = np.sum(np.abs(X_train_raw) > raw_clip)
n_clipped_calib = np.sum(np.abs(X_calib_raw) > raw_clip)
print(f'  Amostras clipadas: treino={n_clipped_train}  calib={n_clipped_calib}')

def representative_dataset():
    for i in range(len(calib_raw_np)):
        yield [calib_raw_np[i:i+1], calib_acf_np[i:i+1]]

print(f'  {len(calib_raw_np)} sinais de calibracao'
      f'  AFIB={y_calib.sum()}  SR={(y_calib==0).sum()}'
      f'  CH={(dom_calib==0).sum()}  PTB={(dom_calib==1).sum()}')

# ── Conversao TFLite INT8 ────────────────────────────────────────────────
print('\nConvertendo para TFLite INT8...')
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations             = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type      = tf.int8
converter.inference_output_type     = tf.int8
tflite_model = converter.convert()
with open(TFLITE_PATH, 'wb') as f: f.write(tflite_model)
print(f'  TFLite INT8: {os.path.getsize(TFLITE_PATH)/1024:.1f} KB  -> {TFLITE_PATH}')

# ── Validacao TFLite ─────────────────────────────────────────────────────
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp_details = interp.get_input_details()
out_details = interp.get_output_details()

print('\nParametros de quantizacao para o firmware ESP32-S3:')
print('-' * 65)
for d in inp_details:
    name = 'input_raw' if 'raw' in d['name'] else 'input_acf'
    print(f"  {name:<20} scale={d['quantization'][0]:.6f}  "
          f"zp={d['quantization'][1]:5d}  dtype={d['dtype'].__name__}")
print(f"  {'output':<20} scale={out_details[0]['quantization'][0]:.6f}  "
      f"zp={out_details[0]['quantization'][1]:5d}  "
      f"dtype={out_details[0]['dtype'].__name__}")
print('-' * 65)
print(f'  Threshold firmware (provisorio, float32): {THRESHOLD_FINAL:.6f}')

# ── Sanity check: fidelidade INT8 vs Keras Float32 ──────────────────────
def quantize(x, scale, zp):   return np.clip(np.round(x/scale+zp), -128, 127).astype(np.int8)
def dequantize(x, scale, zp): return (x.astype(np.float32) - zp) * scale

raw_in  = next(d for d in inp_details if 'raw'  in d['name'])
feat_in = next(d for d in inp_details if 'acf'  in d['name'])

# Testa em 200 amostras do calib
N_CHECK = min(200, len(calib_raw_np))
probs_int8 = np.zeros(N_CHECK, dtype=np.float32)
for i in range(N_CHECK):
    interp.set_tensor(raw_in['index'],
        quantize(calib_raw_np[i:i+1], raw_in['quantization'][0], raw_in['quantization'][1]))
    interp.set_tensor(feat_in['index'],
        quantize(calib_acf_np[i:i+1], feat_in['quantization'][0], feat_in['quantization'][1]))
    interp.invoke()
    out_q = interp.get_tensor(out_details[0]['index'])
    probs_int8[i] = dequantize(out_q, out_details[0]['quantization'][0],
                                out_details[0]['quantization'][1]).flatten()[0]

# Compara com Keras Float32
probs_keras = keras_model.predict(
    [X_calib_raw[:N_CHECK].reshape(-1, N_POINTS, 1), X_calib_acf[:N_CHECK]],
    batch_size=32, verbose=0).flatten()

from scipy.stats import pearsonr
r_fidelity = pearsonr(probs_int8, probs_keras)[0]
print(f'\n  Fidelidade INT8 vs Keras Float32 (n={N_CHECK}): r = {r_fidelity:.4f}')
if r_fidelity < 0.99:
    print(f'  AVISO: fidelidade abaixo do esperado (>=0.99). Verifique outliers.')
else:
    print(f'  OK: fidelidade INT8 preservada.')

# ── Header C provisorio ─────────────────────────────────────────────────
with open(TFLITE_PATH, 'rb') as f: model_data = f.read()
var_name = 'ecg_hybrid_v15_sem_loess_nlm_int8_tflite'
with open(C_HEADER_PATH, 'w') as f:
    f.write(f'// Auto-generated — ECGHybridV15 | {VARIANT_LABEL} | INT8 para ESP32-S3\n')
    f.write('// Split: pool combinado CH+PTB, tres vias 80/5/15 PATIENT-AWARE\n')
    f.write('// Winsorizacao anti-outlier no representative dataset (v3)\n')
    f.write(f'// Holdout patient-aware: N={len(y_ho)}  AFIB={int(y_ho.sum())}  SR={int((y_ho==0).sum())}\n')
    f.write('// Calibracao INT8: conjunto formal 5% do pool combinado\n')
    f.write(f'// Tamanho: {len(model_data)} bytes ({len(model_data)/1024:.1f} KB)\n')
    f.write(f'// Threshold float32 (Youden): {THRESHOLD_FINAL:.6f}  [recalibrado na Cell 40]\n\n')
    f.write('#pragma once\n\n')
    f.write(f'#define ECG_MODEL_VARIANT "{VARIANT_ID}"\n\n')
    f.write(f'#define ECG_HYBRID_V15_SEM_LOESS_NLM_THRESHOLD {THRESHOLD_FINAL:.6f}f  // provisorio\n\n')
    f.write(f'alignas(8) const unsigned char {var_name}[] = {{\n  ')
    hex_vals = [f'0x{b:02x}' for b in model_data]
    f.write(',\n  '.join([', '.join(hex_vals[i:i+16]) for i in range(0, len(hex_vals), 16)]))
    f.write(f'\n}};\nconst unsigned int {var_name}_len = {len(model_data)};\n')
print(f'\nHeader C provisorio gerado: {C_HEADER_PATH}')
print('  (threshold final sera atualizado na Cell 40 apos recalibracao INT8)')


Preparando dados de calibracao INT8 (conjunto formal 5%)...
  Clip anti-outlier RAW: +/-1.1528 mV
    (baseado em p99.9=+1.1528 / p0.1=-0.4360 do treino)
  Amostras clipadas: treino=115984  calib=8462
  1420 sinais de calibracao  AFIB=166  SR=1254  CH=501  PTB=919

Convertendo para TFLite INT8...
INFO:tensorflow:Assets written to: /tmp/tmpwya5_srr/assets


INFO:tensorflow:Assets written to: /tmp/tmpwya5_srr/assets


Saved artifact at '/tmp/tmpwya5_srr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 5000, 1), dtype=tf.float32, name='input_raw'), TensorSpec(shape=(None, 5000), dtype=tf.float32, name='input_acf')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  131431025118096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025122704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025122320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025120016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025121168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025123088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025124048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025123664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025121552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131431025121744: 

/home/jose_alberto/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1790529020.341746  190719 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1790529020.341764  190719 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1790529020.359613  190719 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
/home/jose_alberto/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migr

  TFLite INT8: 189.0 KB  -> /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_int8.tflite

Parametros de quantizacao para o firmware ESP32-S3:
-----------------------------------------------------------------
  input_raw            scale=0.009042  zp=    0  dtype=int8
  input_acf            scale=0.107612  zp=  -30  dtype=int8
  output               scale=0.003906  zp= -128  dtype=int8
-----------------------------------------------------------------
  Threshold firmware (provisorio, float32): 0.791590


I0000 00:00:1790529026.591969  203560 service.cc:152] XLA service 0x7788dc013e50 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790529026.591984  203560 service.cc:160]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1790529026.742986  203560 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



  Fidelidade INT8 vs Keras Float32 (n=200): r = 0.9995
  OK: fidelidade INT8 preservada.

Header C provisorio gerado: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_model.h
  (threshold final sera atualizado na Cell 40 apos recalibracao INT8)


## 17. Recalibração do threshold no modelo TFLite INT8

A quantização INT8 introduz um **shift na distribuição de saída** que invalida
o threshold ótimo (τ) calculado no modelo float32. As causas principais são:

1. **Resolução discreta**: a saída INT8 assume apenas 256 valores (step ≈ 0.0039)
2. **Erro acumulado**: arredondamento em cada camada propaga e desloca a distribuição
3. **Assimetria entre classes**: o erro pode afetar AFIB e SR de forma desigual

**Estratégia**: recalibrar τ diretamente no modelo TFLite INT8 usando os dados
de validação do *best fold* do K-Fold (nunca vistos no treino daquele fold).
O holdout patient-aware permanece **intocado** — usado exclusivamente para avaliação final.

**Justificativa metodológica**: o threshold é um hiperparâmetro do ambiente de
*deployment*, não do modelo. Cada representação numérica (float32, INT8) requer
calibração independente do seu ponto de operação — sem contaminação do holdout.

In [18]:
# =====================================================================
# INFERENCIA TFLITE INT8 NOS DADOS DE VALIDACAO DO BEST FOLD
# =====================================================================
# Os dados best_fold_val_raw/acf/y foram salvos durante o K-Fold
# (Cell 21) quando o melhor fold foi identificado.
# =====================================================================

def tflite_predict_batch(tflite_path, X_raw, X_acf, report_every=200):
    """Inferencia batch no modelo TFLite INT8.
    
    Quantiza entradas, executa inferencia, dequantiza saida.
    Retorna array de probabilidades float32.
    """
    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    
    inp_details = interp.get_input_details()
    out_details = interp.get_output_details()
    
    raw_det = next(d for d in inp_details if 'raw' in d['name'])
    acf_det = next(d for d in inp_details if 'acf' in d['name'])
    out_det = out_details[0]
    
    raw_s, raw_zp = raw_det['quantization']
    acf_s, acf_zp = acf_det['quantization']
    out_s, out_zp = out_det['quantization']
    
    probs = []
    n = len(X_raw)
    for i in range(n):
        raw_q = np.clip(np.round(X_raw[i:i+1] / raw_s + raw_zp),
                        -128, 127).astype(np.int8)
        acf_q = np.clip(np.round(X_acf[i:i+1] / acf_s + acf_zp),
                        -128, 127).astype(np.int8)
        
        interp.set_tensor(raw_det['index'], raw_q)
        interp.set_tensor(acf_det['index'], acf_q)
        interp.invoke()
        
        out_q = interp.get_tensor(out_det['index'])
        prob  = (out_q.astype(np.float32) - out_zp) * out_s
        probs.append(float(prob.flatten()[0]))
        
        if report_every and (i + 1) % report_every == 0:
            print(f"  {i+1:5d}/{n} sinais...")
    
    return np.array(probs)


print(f"{'='*65}")
print(f"  RECALIBRACAO DO THRESHOLD — TFLite INT8")
print(f"{'='*65}")
print(f"  Dados: validacao do best fold {best_fold} (n={len(best_fold_val_y)})")
print(f"    AFIB: {best_fold_val_y.sum():.0f}   SR: {(1-best_fold_val_y).sum():.0f}")
print(f"  Threshold float32 (Youden): {THRESHOLD_FINAL:.6f}")
print()

# ── Inferencia float32 nos dados de validacao (referencia) ───────────────
p_val_float = eval_ds(best_model, best_fold_val_raw, best_fold_val_acf,
                      best_fold_val_y)

# ── Inferencia TFLite INT8 nos mesmos dados ──────────────────────────────
print("Inferencia TFLite INT8 na validacao do best fold...")
p_val_tflite = tflite_predict_batch(TFLITE_PATH,
                                     best_fold_val_raw,
                                     best_fold_val_acf)
print(f"  Concluido: {len(p_val_tflite)} sinais\n")

# ── Diagnostico de divergencia float32 vs INT8 ───────────────────────────
diff = p_val_float - p_val_tflite
mae     = np.mean(np.abs(diff))
max_err = np.max(np.abs(diff))
bias    = diff.mean()
print(f"  Divergencia float32 vs INT8 (validacao best fold):")
print(f"    MAE         : {mae:.6f}")
print(f"    Max |erro|  : {max_err:.6f}")
print(f"    Bias (media): {bias:+.6f}  ({'float > INT8' if bias > 0 else 'INT8 > float'})")
print(f"    Media float : {p_val_float.mean():.6f}")
print(f"    Media INT8  : {p_val_tflite.mean():.6f}")

  RECALIBRACAO DO THRESHOLD — TFLite INT8
  Dados: validacao do best fold 2 (n=2272)
    AFIB: 266   SR: 2006
  Threshold float32 (Youden): 0.791590

Inferencia TFLite INT8 na validacao do best fold...
    200/2272 sinais...
    400/2272 sinais...
    600/2272 sinais...
    800/2272 sinais...
   1000/2272 sinais...
   1200/2272 sinais...
   1400/2272 sinais...
   1600/2272 sinais...
   1800/2272 sinais...
   2000/2272 sinais...
   2200/2272 sinais...
  Concluido: 2272 sinais

  Divergencia float32 vs INT8 (validacao best fold):
    MAE         : 0.010540
    Max |erro|  : 0.231545
    Bias (media): -0.005907  (INT8 > float)
    Media float : 0.243814
    Media INT8  : 0.249721


In [19]:
# ── Threshold otimo Youden no modelo TFLite INT8 ─────────────────────────
fpr_q, tpr_q, thr_q = roc_curve(best_fold_val_y, p_val_tflite)
youden_q = tpr_q - fpr_q
THRESH_TFLITE_YOUDEN = float(thr_q[np.argmax(youden_q)])

# ── Snap ao grid discreto INT8 ───────────────────────────────────────────
# A saida TFLite INT8 so assume valores discretos: (q - zp) * scale
# Restringir o threshold a um desses valores garante comportamento
# identico entre Python e firmware ESP32-S3.
out_scale = out_details[0]['quantization'][0]
out_zp    = out_details[0]['quantization'][1]
grid_vals = (np.arange(-128, 128).astype(np.float32) - out_zp) * out_scale

# Busca exaustiva no grid discreto (balanced_accuracy = (Sens+Spec)/2)
best_ba   = -1
best_grid = THRESH_TFLITE_YOUDEN
for tau_g in grid_vals:
    m_g = metrics(p_val_tflite, best_fold_val_y, tau_g)
    ba  = (m_g['sens'] + m_g['spec']) / 2.0
    if ba > best_ba:
        best_ba   = ba
        best_grid = float(tau_g)

THRESH_TFLITE_GRID = best_grid

print(f"\n{'─'*65}")
print(f"  COMPARACAO DE THRESHOLDS")
print(f"{'─'*65}")
print(f"  {'Metodo':<35} {'tau':>10} {'Sens':>7} {'Spec':>7} {'F1':>7}")
print(f"  {'─'*65}")

for label, tau in [
    ('Float32 Youden (original)',    THRESHOLD_FINAL),
    ('INT8 Youden (continuo)',       THRESH_TFLITE_YOUDEN),
    ('INT8 grid discreto (firmware)', THRESH_TFLITE_GRID),
]:
    m = metrics(p_val_tflite, best_fold_val_y, tau)
    print(f"  {label:<35} {tau:>10.6f} {m['sens']:>7.4f} {m['spec']:>7.4f} {m['f1']:>7.4f}")

# ── Selecao final: threshold para firmware ────────────────────────────────
THRESH_FIRMWARE = THRESH_TFLITE_GRID
print(f"\n  >>> Threshold firmware (INT8 recalibrado): {THRESH_FIRMWARE:.6f}")

# ── Salvar threshold recalibrado ─────────────────────────────────────────
THRESH_TFLITE_PATH = f'{BASE}/ecg_hybrid_v15_sem_loess_nlm_threshold_tflite.pkl'
with open(THRESH_TFLITE_PATH, 'wb') as f:
    pickle.dump({
        'tau_float32_youden':    THRESHOLD_FINAL,
        'tau_int8_youden':       THRESH_TFLITE_YOUDEN,
        'tau_int8_grid':         THRESH_TFLITE_GRID,
        'tau_firmware':          THRESH_FIRMWARE,
        'out_scale':             float(out_scale),
        'out_zp':                int(out_zp),
        'calib_fold':            best_fold,
        'calib_n':               len(best_fold_val_y),
    }, f)
print(f"  Salvo: {THRESH_TFLITE_PATH}")


─────────────────────────────────────────────────────────────────
  COMPARACAO DE THRESHOLDS
─────────────────────────────────────────────────────────────────
  Metodo                                     tau    Sens    Spec      F1
  ─────────────────────────────────────────────────────────────────
  Float32 Youden (original)             0.791590  0.9887  0.9865  0.9460
  INT8 Youden (continuo)                0.816406  0.9887  0.9930  0.9687
  INT8 grid discreto (firmware)         0.812500  0.9887  0.9930  0.9687

  >>> Threshold firmware (INT8 recalibrado): 0.812500
  Salvo: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ecg_hybrid_v15_sem_loess_nlm_threshold_tflite.pkl


In [20]:
# ── Histograma de probabilidades float32 vs INT8 ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'Distribuicao de probabilidades — Float32 vs TFLite INT8\n'
    f'Validacao best fold {best_fold} (n={len(best_fold_val_y)})',
    fontsize=11, y=1.02)

for ax, (probs, title) in zip(axes, [
    (p_val_float,  'Float32 (PyTorch)'),
    (p_val_tflite, 'INT8 (TFLite)'),
]):
    mask_sr   = best_fold_val_y == 0
    mask_afib = best_fold_val_y == 1
    
    ax.hist(probs[mask_sr],   bins=60, alpha=0.6, color=C_SR,   label='SR',   density=True)
    ax.hist(probs[mask_afib], bins=60, alpha=0.6, color=C_AFIB, label='AFIB', density=True)
    
    if 'Float' in title:
        ax.axvline(THRESHOLD_FINAL, color='green', ls='--', lw=1.5,
                   label=f'tau float = {THRESHOLD_FINAL:.4f}')
    else:
        ax.axvline(THRESHOLD_FINAL, color='green', ls='--', lw=1.2, alpha=0.5,
                   label=f'tau float = {THRESHOLD_FINAL:.4f}')
        ax.axvline(THRESH_FIRMWARE, color='red', ls='-', lw=1.8,
                   label=f'tau INT8 = {THRESH_FIRMWARE:.4f}')
    
    ax.set_xlabel('Probabilidade AFIB', fontsize=9)
    ax.set_ylabel('Densidade', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_xlim(-0.05, 1.05)

plt.tight_layout()
hist_path = f'{BASE}/ecg_v15_sem_loess_nlm_hist_float_vs_int8.png'
plt.savefig(hist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Histograma salvo: {hist_path}')

Histograma salvo: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ecg_v15_sem_loess_nlm_hist_float_vs_int8.png


/tmp/ipykernel_190719/3979469118.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# ── Regerar header C com threshold recalibrado ────────────────────────────
print(f"\nAtualizando header C com threshold INT8 recalibrado...")
with open(TFLITE_PATH, 'rb') as f: model_data = f.read()
var_name = 'ecg_hybrid_v15_sem_loess_nlm_int8_tflite'
with open(C_HEADER_PATH, 'w') as f:
    f.write(f'// Auto-generated — ECGHybridV15 | {VARIANT_LABEL} | INT8 para ESP32-S3\n')
    f.write('// Split: patient-aware (StratifiedGroupKFold) | treino 80,7% / calib 5,0% / holdout 14,3%\n')
    f.write('// Ablation: sem CORAL + SEM LOESS+NLM (Ablation: LOESS e NLM removidos (apenas Butterworth))\n')
    f.write(f'// Tamanho: {len(model_data)} bytes ({len(model_data)/1024:.1f} KB)\n')
    f.write(f'// Threshold float32 (Youden):  {THRESHOLD_FINAL:.6f}\n')
    f.write(f'// Threshold INT8 recalibrado:  {THRESH_FIRMWARE:.8f}\n')
    f.write(f'// Recalibracao: Youden + grid discreto INT8 na validacao fold {best_fold}\n\n')
    f.write('#pragma once\n\n')
    f.write(f'#define ECG_MODEL_VARIANT "{VARIANT_ID}"\n\n')
    f.write(f'// Threshold recalibrado para o modelo TFLite INT8\n')
    f.write(f'// (Youden Index otimizado no grid discreto da saida quantizada)\n')
    f.write(f'#define ECG_HYBRID_V15_SEMCORAL_THRESHOLD {THRESH_FIRMWARE:.8f}f\n')
    _q_thr = int(round(THRESH_FIRMWARE / out_details[0]['quantization'][0]
                       + out_details[0]['quantization'][1]))
    f.write(f'// Limiar inteiro na saida INT8: classificar AFIB se (int8_out >= ECG_THRESHOLD_Q)\n')
    f.write(f'#define ECG_THRESHOLD_Q ({_q_thr})\n\n')
    f.write(f'// Threshold original float32 (referencia — NAO usar no firmware)\n')
    f.write(f'// #define ECG_HYBRID_V15_SEMCORAL_THRESHOLD_FLOAT {THRESHOLD_FINAL:.6f}f\n\n')
    f.write(f'alignas(8) const unsigned char {var_name}[] = {{\n  ')
    hex_vals = [f'0x{b:02x}' for b in model_data]
    f.write(',\n  '.join([', '.join(hex_vals[i:i+16]) for i in range(0, len(hex_vals), 16)]))
    f.write(f'\n}};\nconst unsigned int {var_name}_len = {len(model_data)};\n')
print(f'  Header C atualizado: {C_HEADER_PATH}')
print(f'  #define THRESHOLD = {THRESH_FIRMWARE:.8f}f   |   ECG_THRESHOLD_Q = {_q_thr}')


Atualizando header C com threshold INT8 recalibrado...
  Header C atualizado: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_model.h
  #define THRESHOLD = 0.81250000f   |   ECG_THRESHOLD_Q = 80


In [22]:
print(f"{'='*60}")
print(f"  ECGHybridV15 SEM CORAL + SEM LOESS+NLM (ablation) -- Pipeline concluido")
print(f"{'='*60}")
for path, desc in [
    (FINAL_MODEL_PATH,  'Pesos PyTorch (.pt)'),
    (SCALER_PATH,       'StandardScaler (.pkl)'),
    (THRESH_PATH,       'Threshold float32 (.pkl)'),
    (THRESH_TFLITE_PATH,'Threshold INT8 recalibrado (.pkl)'),
    (HOLDOUT_CSV_PATH,  'Planilha holdout patient-aware (.csv)'),
    (TFLITE_PATH,       'TFLite INT8 (.tflite)  <- firmware'),
    (C_HEADER_PATH,     'Header C (.h)           <- ESP32-S3'),
    (ROC_PATH,          'Curva ROC (.png)'),
    (WAVE_PLOT_PATH,    'Plot ondas bruto/filtrado (.png)'),
    (RESULTS_PATH,      'Diagnostico treino (.png)'),
]:
    ok = 'OK' if os.path.exists(path) else '--'
    print(f'  [{ok}] {desc}')
    print(f'       {path}')

print(f"\n  Holdout patient-aware (≈14,3% do pool)  n={len(y_ho)}  AFIB={y_ho.sum()}  SR={(y_ho==0).sum()}")
print(f"\n  THRESHOLDS:")
print(f"    tau float32 (Youden K-Fold)      : {THRESHOLD_FINAL:.6f}")
print(f"    tau INT8 recalibrado (firmware)   : {THRESH_FIRMWARE:.6f}")
print(f"    delta                             : {THRESH_FIRMWARE - THRESHOLD_FINAL:+.6f}")
print(f"\n  HOLDOUT — modelo float32 com tau_float:")
print(f"    AUC total  : {m_ho_total['auc']:.4f}")
print(f"    Sens total : {m_ho_total['sens']:.4f}")
print(f"    Spec total : {m_ho_total['spec']:.4f}")
print(f"    F1   total : {m_ho_total['f1']:.4f}")

# ── Holdout com TFLite INT8 e threshold recalibrado ──────────────────────
print(f"\n  HOLDOUT — TFLite INT8 com tau_int8 recalibrado:")
print(f"  (inferencia TFLite INT8 no holdout para comparacao final)")
p_ho_tflite = tflite_predict_batch(TFLITE_PATH, X_ho_raw, X_ho_acf,
                                    report_every=500)

m_ho_tflite_new = metrics(p_ho_tflite, y_ho, THRESH_FIRMWARE)
m_ho_tflite_old = metrics(p_ho_tflite, y_ho, THRESHOLD_FINAL)

print(f"\n  {'Cenario':<40} {'Sens':>7} {'Spec':>7} {'F1':>7} {'AUC':>7}")
print(f"  {'─'*65}")
print(f"  {'Float32 + tau_float':<40} "
      f"{m_ho_total['sens']:>7.4f} {m_ho_total['spec']:>7.4f} "
      f"{m_ho_total['f1']:>7.4f} {m_ho_total['auc']:>7.4f}")
print(f"  {'INT8 + tau_float (sem recalibrar)':<40} "
      f"{m_ho_tflite_old['sens']:>7.4f} {m_ho_tflite_old['spec']:>7.4f} "
      f"{m_ho_tflite_old['f1']:>7.4f} {m_ho_tflite_old['auc']:>7.4f}")
print(f"  {'INT8 + tau_int8 (recalibrado)':<40} "
      f"{m_ho_tflite_new['sens']:>7.4f} {m_ho_tflite_new['spec']:>7.4f} "
      f"{m_ho_tflite_new['f1']:>7.4f} {m_ho_tflite_new['auc']:>7.4f}")

print(f"\n  AUC CH  : {m_ho_ch['auc']:.4f}")
print(f"  AUC PTB : {m_ho_ptb['auc']:.4f}")

/home/jose_alberto/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  ECGHybridV15 SEM CORAL + SEM LOESS+NLM (ablation) -- Pipeline concluido
  [OK] Pesos PyTorch (.pt)
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_final.pt
  [OK] StandardScaler (.pkl)
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_scaler.pkl
  [OK] Threshold float32 (.pkl)
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_threshold.pkl
  [OK] Threshold INT8 recalibrado (.pkl)
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ecg_hybrid_v15_sem_loess_nlm_threshold_tflite.pkl
  [OK] Planilha holdout patient-aware (.csv)
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_holdout.csv
  [OK] TFLite INT8 (.tflite)  <- firmware
       /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_int8.tflite
  [OK] Header C (.h)           <- ESP32-S3
  

In [23]:
# ── Curvas ROC separadas com IC bootstrap (n=1000) ───────────────────────
from sklearn.utils import resample

N_BOOT   = 1000
ALPHA    = 0.05   # IC 95%
RNG_BOOT = np.random.RandomState(SEED)

def roc_bootstrap(probs, labels, n_boot=N_BOOT, rng=RNG_BOOT):
    """Retorna (fpr_mean, tpr_grid, tpr_lower, tpr_upper, auc_mean, auc_ci)."""
    fpr_grid  = np.linspace(0, 1, 300)
    tpr_boot  = []
    aucs_boot = []
    for _ in range(n_boot):
        pb, lb = resample(probs, labels, random_state=rng)
        if len(np.unique(lb)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(lb, pb)
        tpr_boot.append(np.interp(fpr_grid, fpr_b, tpr_b))
        aucs_boot.append(sk_auc(fpr_b, tpr_b))
    tpr_boot  = np.array(tpr_boot)
    aucs_boot = np.array(aucs_boot)
    tpr_lo = np.percentile(tpr_boot, 100 * ALPHA/2,     axis=0)
    tpr_hi = np.percentile(tpr_boot, 100 * (1-ALPHA/2), axis=0)
    auc_lo = np.percentile(aucs_boot, 100 * ALPHA/2)
    auc_hi = np.percentile(aucs_boot, 100 * (1-ALPHA/2))
    fpr_c, tpr_c, _ = roc_curve(labels, probs)
    return fpr_c, tpr_c, fpr_grid, tpr_lo, tpr_hi, sk_auc(fpr_c, tpr_c), (auc_lo, auc_hi)

print('Calculando bootstrap ROC (n=1000)...')
fpr_ch,  tpr_ch,  fg_ch,  tlo_ch,  thi_ch,  auc_ch,  ci_ch  = roc_bootstrap(p_ho_all[mask_ch],  y_ho[mask_ch])
fpr_ptb, tpr_ptb, fg_ptb, tlo_ptb, thi_ptb, auc_ptb, ci_ptb = roc_bootstrap(p_ho_all[mask_ptb], y_ho[mask_ptb])
fpr_tot, tpr_tot, fg_tot, tlo_tot, thi_tot, auc_tot, ci_tot  = roc_bootstrap(p_ho_all,           y_ho)
print('  OK')

# ── Configuracoes dos 3 paineis ────────────────────────────────────────────
panels = [
    dict(fpr=fpr_ch,  tpr=tpr_ch,  fg=fg_ch,  tlo=tlo_ch,  thi=thi_ch,
         auc=auc_ch,  ci=ci_ch,  color=C_SR,
         m=m_ho_ch,   label='CH holdout',
         n=int(mask_ch.sum()),  title='CH Holdout'),
    dict(fpr=fpr_ptb, tpr=tpr_ptb, fg=fg_ptb, tlo=tlo_ptb, thi=thi_ptb,
         auc=auc_ptb, ci=ci_ptb, color=C_AFIB,
         m=m_ho_ptb,  label='PTB holdout',
         n=int(mask_ptb.sum()), title='PTB-XL Holdout'),
    dict(fpr=fpr_tot, tpr=tpr_tot, fg=fg_tot, tlo=tlo_tot, thi=thi_tot,
         auc=auc_tot, ci=ci_tot, color='#1D9E75',
         m=m_ho_total, label='Total holdout',
         n=len(y_ho),             title='Total (CH + PTB-XL)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f'Curvas ROC (float32) — {VARIANT_LABEL} | '
    f'tau = {THRESHOLD_FINAL:.4f} | IC 95% bootstrap (n={N_BOOT})',
    fontsize=11, y=1.02)

for ax, p in zip(axes, panels):
    ax.fill_between(p['fg'], p['tlo'], p['thi'],
                    alpha=0.18, color=p['color'], label='IC 95% bootstrap')
    ax.plot(p['fpr'], p['tpr'], color=p['color'], lw=2.2,
            label=f"AUC = {p['auc']:.4f}  [{p['ci'][0]:.4f}–{p['ci'][1]:.4f}]")
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.35)
    op_x = 1 - p['m']['spec']
    op_y = p['m']['sens']
    ax.scatter([op_x], [op_y], color=p['color'], s=100, zorder=6,
               edgecolors='white', linewidths=1.0, marker='D')
    ax.annotate(
        f"Sens={op_y:.3f}\nSpec={p['m']['spec']:.3f}",
        xy=(op_x, op_y), xytext=(10, -22),
        textcoords='offset points', fontsize=7.5, color=p['color'],
        arrowprops=dict(arrowstyle='->', color=p['color'], lw=0.8))
    ax.set_title(f"{p['title']}  (n={p['n']})", fontsize=10)
    ax.set_xlabel('FPR  (1 – Especificidade)', fontsize=9)
    ax.set_ylabel('TPR  (Sensibilidade)',       fontsize=9)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.22)
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
roc_sep_path = ROC_PATH.replace('.png', '_separadas.png')
plt.savefig(roc_sep_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nFigura salva: {roc_sep_path}')
for p in panels:
    print(f"  {p['title']:<22} AUC={p['auc']:.4f}  IC95=[{p['ci'][0]:.4f}, {p['ci'][1]:.4f}]")

Calculando bootstrap ROC (n=1000)...
  OK

Figura salva: /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3//ablation_loess_nlm_roc_separadas.png
  CH Holdout             AUC=0.9984  IC95=[0.9972, 0.9993]
  PTB-XL Holdout         AUC=0.9960  IC95=[0.9938, 0.9980]
  Total (CH + PTB-XL)    AUC=0.9971  IC95=[0.9958, 0.9982]


/tmp/ipykernel_190719/2961437574.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# Ponto de operação alternativo (Sens >= 0.99) — OPCIONAL, não usado nas tabelas principais.
# ═══════════════════════════════════════════════════════════════
# Recalibração do threshold para sensibilidade alvo ≥ 0.99
# Metodologia: threshold escolhido no conjunto de calibração INT8
# (não no holdout); avaliação final no holdout com o novo threshold.
# ═══════════════════════════════════════════════════════════════
import numpy as np, pickle
from sklearn.metrics import roc_auc_score, confusion_matrix

TARGET_SENS = 0.99
BOOTSTRAP_N = 1000
SEED_BOOT   = 42

# ── Detecta as probabilidades INT8 no calib e no holdout ────────────────
# (nomes verificados no notebook do Ablation #1)
p_calib = None
for name in ["probs_int8_calib", "p_calib_tflite", "p_calib_int8"]:
    if name in globals():
        p_calib = np.asarray(globals()[name]).flatten()
        break
if p_calib is None:
    # Se não existe ainda, computa agora usando o interpreter INT8
    print("Rodando inferência INT8 no calib para recalibrar threshold...")
    p_calib = np.zeros(len(X_calib_raw), dtype=np.float32)
    raw_in_d  = next(d for d in interp.get_input_details() if 'raw' in d['name'])
    acf_in_d  = next(d for d in interp.get_input_details() if 'acf' in d['name'])
    out_d     = interp.get_output_details()[0]
    raw_shape = tuple(raw_in_d['shape'])
    acf_shape = tuple(acf_in_d['shape'])
    def _q(x, s, z): return np.clip(np.round(x/s+z), -128, 127).astype(np.int8)
    def _d(x, s, z): return (x.astype(np.float32) - z) * s
    for i in range(len(X_calib_raw)):
        raw_q = _q(X_calib_raw[i], raw_in_d['quantization'][0], raw_in_d['quantization'][1]).reshape(raw_shape)
        acf_q = _q(X_calib_acf[i], acf_in_d['quantization'][0], acf_in_d['quantization'][1]).reshape(acf_shape)
        interp.set_tensor(raw_in_d['index'], raw_q)
        interp.set_tensor(acf_in_d['index'], acf_q)
        interp.invoke()
        out_q = interp.get_tensor(out_d['index'])
        p_calib[i] = _d(out_q, out_d['quantization'][0], out_d['quantization'][1]).flatten()[0]
    print(f"  p_calib computado: N={len(p_calib)}")

# Vetores do holdout
p_ho = np.asarray(p_ho_tflite).flatten()
y_c  = np.asarray(y_calib).astype(int)
y_h  = np.asarray(y_ho).astype(int)

# ── Busca do threshold no calib ─────────────────────────────────────────
# Estratégia: entre todos os thresholds candidatos (probabilidades observadas),
# achar o MAIOR que ainda garante Sens ≥ TARGET_SENS.
# Isso maximiza especificidade dado o vínculo de sensibilidade.
thresholds = np.unique(p_calib)          # todos os candidatos possíveis
thresholds = np.sort(thresholds)         # do menor ao maior

best_tau  = None
best_spec = -1
best_sens = None
for tau_c in thresholds:
    yhat = (p_calib >= tau_c).astype(int)
    tn, fp, fn_, tp = confusion_matrix(y_c, yhat, labels=[0,1]).ravel()
    sens = tp / (tp + fn_) if (tp + fn_) else 0
    spec = tn / (tn + fp)  if (tn + fp)  else 0
    if sens >= TARGET_SENS and spec > best_spec:
        best_tau, best_sens, best_spec = float(tau_c), sens, spec

if best_tau is None:
    print(f"AVISO: nenhum threshold no calib atinge Sens >= {TARGET_SENS}")
    print(f"  Sens máxima no calib: {(p_calib >= p_calib.min()).mean():.4f}")
else:
    print(f"=== Threshold recalibrado no CALIB ===")
    print(f"  Alvo: Sens >= {TARGET_SENS}")
    print(f"  Threshold escolhido: tau_new = {best_tau:.6f}")
    print(f"  No calib: Sens={best_sens:.4f}  Spec={best_spec:.4f}")

# ── Avaliação no holdout com threshold antigo vs novo ───────────────────
def compute_metrics(p, y, tau):
    yhat = (p >= tau).astype(int)
    tn, fp, fn_, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    sens = tp / (tp + fn_) if (tp + fn_) else np.nan
    spec = tn / (tn + fp)  if (tn + fp)  else np.nan
    ppv  = tp / (tp + fp)  if (tp + fp)  else np.nan
    npv  = tn / (tn + fn_) if (tn + fn_) else np.nan
    f1   = 2*tp / (2*tp + fp + fn_) if (2*tp + fp + fn_) else np.nan
    acc  = (tp + tn) / len(y)
    auc  = roc_auc_score(y, p)
    return {"AUC": auc, "Sensitivity": sens, "Specificity": spec,
            "PPV": ppv, "NPV": npv, "F1-Score": f1, "Accuracy": acc,
            "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn_)}

def bootstrap_ci(p, y, tau, n=BOOTSTRAP_N, seed=SEED_BOOT):
    rng = np.random.RandomState(seed)
    keys = ["AUC", "Sensitivity", "Specificity", "PPV", "NPV", "F1-Score", "Accuracy"]
    boot = {k: [] for k in keys}
    N = len(y)
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2: continue
        m = compute_metrics(p[idx], y[idx], tau)
        for k in keys:
            boot[k].append(m[k])
    return {k: (float(np.nanpercentile(boot[k], 2.5)),
                float(np.nanpercentile(boot[k], 97.5))) for k in keys}

# Threshold antigo
tau_old = float(THRESH_FIRMWARE)  # tau INT8 do firmware
m_old  = compute_metrics(p_ho, y_h, tau_old)
ci_old = bootstrap_ci(p_ho, y_h, tau_old)

m_new  = compute_metrics(p_ho, y_h, best_tau)
ci_new = bootstrap_ci(p_ho, y_h, best_tau)

# ── Impressão comparativa ───────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  COMPARAÇÃO NO HOLDOUT (N={len(y_h)})")
print(f"  Threshold firmware (INT8 grid): tau = {tau_old:.6f}")
print(f"  Threshold novo (Sens>=0.99): tau = {best_tau:.6f}")
print(f"{'='*80}")
print(f"  {'Métrica':<14}{'Antigo':>10}   {'95% CI':<26}   "
      f"{'Novo':>10}   {'95% CI':<26}")
print(f"  {'-'*14}{'-'*10}   {'-'*26}   {'-'*10}   {'-'*26}")
for k in ["AUC", "Sensitivity", "Specificity", "PPV", "NPV", "F1-Score", "Accuracy"]:
    lo_o, hi_o = ci_old[k]
    lo_n, hi_n = ci_new[k]
    print(f"  {k:<14}{m_old[k]:>10.4f}   [{lo_o:.4f}, {hi_o:.4f}]   "
          f"{m_new[k]:>10.4f}   [{lo_n:.4f}, {hi_n:.4f}]")
print(f"{'='*80}")
print(f"  Matriz de confusão (novo threshold):")
print(f"    TP={m_new['TP']}  TN={m_new['TN']}  FP={m_new['FP']}  FN={m_new['FN']}")
print(f"  AFIB perdidos: {m_new['FN']} (antes: {m_old['FN']})")
print(f"  FPs adicionais: {m_new['FP'] - m_old['FP']:+d}")
print(f"{'='*80}")

# ── Salva o novo threshold em arquivo separado ──────────────────────────
new_thresh_path = f"{BASE}/ablation_loess_nlm_threshold_sens99.pkl"
with open(new_thresh_path, 'wb') as f:
    pickle.dump({
        "threshold_int8": best_tau,
        "target_sensitivity": TARGET_SENS,
        "calibration_set_size": len(p_calib),
        "calib_sens": best_sens,
        "calib_spec": best_spec,
        "holdout_sens": m_new['Sensitivity'],
        "holdout_spec": m_new['Specificity'],
        "notes": "Threshold recalibrated to target sensitivity >= 0.99 for screening scenario",
    }, f)
print(f"\nThreshold novo salvo: {new_thresh_path}")
print(f"(o threshold original permanece intacto em ablation_loess_nlm_threshold.pkl)")

# ── Formato LaTeX pronto para a tabela do artigo ────────────────────────
print(f"\n=== Formato LaTeX (operating point com Sens >= 0.99) ===")
for k in ["AUC", "Sensitivity", "Specificity", "PPV", "NPV", "F1-Score", "Accuracy"]:
    lo, hi = ci_new[k]
    print(f"  {k:<14} & {m_new[k]:.4f} & [{lo:.4f}, {hi:.4f}] \\\\")

Rodando inferência INT8 no calib para recalibrar threshold...
  p_calib computado: N=1420
=== Threshold recalibrado no CALIB ===
  Alvo: Sens >= 0.99
  Threshold escolhido: tau_new = 0.664062
  No calib: Sens=0.9940  Spec=0.9745

  COMPARAÇÃO NO HOLDOUT (N=4025)
  Threshold firmware (INT8 grid): tau = 0.812500
  Threshold novo (Sens>=0.99): tau = 0.664062
  Métrica           Antigo   95% CI                             Novo   95% CI                    
  ------------------------   --------------------------   ----------   --------------------------
  AUC               0.9972   [0.9960, 0.9982]       0.9972   [0.9960, 0.9982]
  Sensitivity       0.9618   [0.9423, 0.9777]       0.9873   [0.9767, 0.9959]
  Specificity       0.9879   [0.9843, 0.9912]       0.9676   [0.9621, 0.9734]
  PPV               0.9133   [0.8872, 0.9363]       0.8017   [0.7691, 0.8345]
  NPV               0.9949   [0.9923, 0.9969]       0.9983   [0.9968, 0.9994]
  F1-Score          0.9369   [0.9209, 0.9514]       0.88

In [25]:
# ═══════════════════════════════════════════════════════════════════════
# MÉTRICAS FINAIS INT8 NO HOLDOUT — CÉLULA PADRONIZADA (todas as variantes)
# Gera, a partir das variáveis do notebook (nada digitado à mão):
#   - matriz de confusão e 7 métricas INT8 com IC 95% bootstrap
#   - metrics_final.json
#   - holdout_probs_int8.csv  (probabilidade INT8 por registro; referência
#                              para comparação sinal a sinal com o ESP32-S3)
#   - roc_int8.png            (ROC INT8 com ponto de operação do firmware)
# ═══════════════════════════════════════════════════════════════════════
import json, hashlib
from datetime import datetime
from sklearn.metrics import confusion_matrix

N_BOOT_STD = 1000
SEED_STD   = 42
TAU_FW     = float(THRESH_FIRMWARE)          # τ INT8 do firmware (grade discreta)
p_int8     = np.asarray(p_ho_tflite, dtype=float).ravel()
y_true     = np.asarray(y_ho, dtype=int).ravel()
dom_arr    = np.asarray(dom_ho, dtype=int).ravel()
assert len(p_int8) == len(y_true) == len(dom_arr) == 4025, 'Holdout inesperado'

KEYS = ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1-Score', 'Accuracy']

def compute_std(p, y, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {'AUC': roc_auc_score(y, p),
            'Sensitivity': tp / (tp + fn), 'Specificity': tn / (tn + fp),
            'PPV': tp / (tp + fp) if (tp + fp) else np.nan,
            'NPV': tn / (tn + fn) if (tn + fn) else np.nan,
            'F1-Score': 2 * tp / (2 * tp + fp + fn),
            'Accuracy': (tp + tn) / len(y),
            'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)}

def bootstrap_std(p, y, thr, n=N_BOOT_STD, seed=SEED_STD):
    rng = np.random.RandomState(seed)
    boot = {k: [] for k in KEYS}
    for _ in range(n):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        m = compute_std(p[idx], y[idx], thr)
        for k in KEYS:
            boot[k].append(m[k])
    return {k: [float(np.nanpercentile(boot[k], 2.5)),
                float(np.nanpercentile(boot[k], 97.5))] for k in KEYS}

pt = compute_std(p_int8, y_true, TAU_FW)
ci = bootstrap_std(p_int8, y_true, TAU_FW)
auc_dom = {name: float(roc_auc_score(y_true[dom_arr == d], p_int8[dom_arr == d]))
           for d, name in [(0, 'CH'), (1, 'PTB')]}

# ── Impressão no formato das tabelas do paper ────────────────────────────
print('=' * 68)
print(f'  {VARIANT_LABEL}')
print(f'  LiteRT INT8 | tau firmware = {TAU_FW:.6f} | n = {len(y_true)} '
      f'(AFIB={int(y_true.sum())}, SR={int((y_true == 0).sum())})')
print('=' * 68)
n_afib, n_sr = pt['TP'] + pt['FN'], pt['FP'] + pt['TN']
print(f"  AFIB: TP={pt['TP']} ({100*pt['TP']/n_afib:.2f}%)  FN={pt['FN']} ({100*pt['FN']/n_afib:.2f}%)")
print(f"  SR  : FP={pt['FP']} ({100*pt['FP']/n_sr:.2f}%)  TN={pt['TN']} ({100*pt['TN']/n_sr:.2f}%)")
print(f'\n  Formato LaTeX (bootstrap n={N_BOOT_STD}, seed={SEED_STD}):')
for k in KEYS:
    print(f'  {k:<12} & {pt[k]:.4f} & [{ci[k][0]:.4f}, {ci[k][1]:.4f}] \\\\')
print(f"\n  AUC CH = {auc_dom['CH']:.4f}   AUC PTB = {auc_dom['PTB']:.4f}")

# ── metrics_final.json gerado a partir das variáveis ─────────────────────
record = {
    'variant': VARIANT_LABEL, 'variant_id': VARIANT_ID,
    'split': 'patient-aware (StratifiedGroupKFold)',
    'timestamp': datetime.now().isoformat(),
    'holdout': {'n_total': int(len(y_true)), 'n_afib': int(y_true.sum()),
                'n_sr': int((y_true == 0).sum())},
    'thresholds': {'tau_float32_youden': float(THRESHOLD_FINAL),
                   'tau_int8_firmware': TAU_FW},
    'float32_with_tau_float': {k: float(m_ho_total[k]) for k in ['auc', 'sens', 'spec', 'f1']},
    'int8_with_tau_firmware': {k: float(pt[k]) for k in KEYS},
    'int8_ci95': ci,
    'int8_confusion': {k: pt[k] for k in ['TP', 'FN', 'FP', 'TN']},
    'int8_auc_by_domain': auc_dom,
    'bootstrap': {'n': N_BOOT_STD, 'seed': SEED_STD, 'method': 'percentile'},
    'best_fold': int(best_fold),
    'determinism': {'seed': SEED, 'cudnn_deterministic': torch.backends.cudnn.deterministic,
                    'cudnn_benchmark': torch.backends.cudnn.benchmark,
                    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
                    'torch': torch.__version__},
    'sha256': {'model_pt': hashlib.sha256(open(FINAL_MODEL_PATH, 'rb').read()).hexdigest(),
               'tflite': hashlib.sha256(open(TFLITE_PATH, 'rb').read()).hexdigest()},
}
with open(f'{BASE}/metrics_final.json', 'w') as f:
    json.dump(record, f, indent=2)

# ── Probabilidades INT8 por registro (referência para o firmware) ────────
q_out = np.round(p_int8 / out_details[0]['quantization'][0]
                 + out_details[0]['quantization'][1]).astype(int)
pd.DataFrame({'filename': [os.path.basename(str(x)) for x in fn_ho],
              'dataset': np.where(dom_arr == 0, 'CH', 'PTB'),
              'label_int': y_true, 'int8_out': q_out, 'p_int8': p_int8,
              'pred_int8': (p_int8 >= TAU_FW).astype(int)}
             ).to_csv(f'{BASE}/holdout_probs_int8.csv', index=False)

# ── ROC INT8 com ponto de operação do firmware ───────────────────────────
fpr_q8, tpr_q8, _ = roc_curve(y_true, p_int8)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr_q8, tpr_q8, lw=2, color=C_SR,
        label=f"AUC = {pt['AUC']:.4f} [{ci['AUC'][0]:.4f}, {ci['AUC'][1]:.4f}]")
ax.plot(1 - pt['Specificity'], pt['Sensitivity'], 'o', ms=9, color=C_AFIB,
        label=f"tau = {TAU_FW:.6f}  (Sens={pt['Sensitivity']:.4f}, Spec={pt['Specificity']:.4f})")
ax.plot([0, 1], [0, 1], '--', color='gray', lw=0.8)
ax.set_xlabel('FPR (1 - Specificity)'); ax.set_ylabel('TPR (Sensitivity)')
ax.set_title(f'ROC LiteRT INT8 — {VARIANT_LABEL}\nHoldout patient-aware (n={len(y_true)})', fontsize=10)
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{BASE}/roc_int8.png', dpi=150); plt.show()

print(f"\n  sha256 tflite: {record['sha256']['tflite'][:16]}...  (igual entre execucoes = treino reproduzivel)")
print(f'\n  Salvos em {BASE}:')
print('    metrics_final.json | holdout_probs_int8.csv | roc_int8.png')


  Ablation #1 - No-LOESS/NLM (Butterworth only)
  LiteRT INT8 | tau firmware = 0.812500 | n = 4025 (AFIB=471, SR=3554)
  AFIB: TP=453 (96.18%)  FN=18 (3.82%)
  SR  : FP=43 (1.21%)  TN=3511 (98.79%)

  Formato LaTeX (bootstrap n=1000, seed=42):
  AUC          & 0.9972 & [0.9960, 0.9982] \\
  Sensitivity  & 0.9618 & [0.9423, 0.9777] \\
  Specificity  & 0.9879 & [0.9843, 0.9912] \\
  PPV          & 0.9133 & [0.8872, 0.9363] \\
  NPV          & 0.9949 & [0.9923, 0.9969] \\
  F1-Score     & 0.9369 & [0.9209, 0.9514] \\
  Accuracy     & 0.9848 & [0.9814, 0.9883] \\

  AUC CH = 0.9982   AUC PTB = 0.9964

  sha256 tflite: a7582a6d1dd77275...  (igual entre execucoes = treino reproduzivel)

  Salvos em /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3/:
    metrics_final.json | holdout_probs_int8.csv | roc_int8.png


/tmp/ipykernel_190719/3626127281.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f'{BASE}/roc_int8.png', dpi=150); plt.show()
